# CS-781 Phase 5 — Legal Statute Identification (LSI)

**Goal:** build a reproducible multi-label statute prediction pipeline using your existing Phase-1 Legal Language Models and the official Phase-5 LSI data.

## What this notebook does
- Finds `Phase_5_LSI/Train/train.json`, `Train/dev.json`, `Test/test.json` in attached Kaggle inputs, or downloads the same official files from `Exploration-Lab/CS-781-Capstone`.
- Finds compatible A/B/C/D Phase-1 checkpoints and the exact `LEGAL_40K` tokenizer; it reports missing/incompatible architectures instead of pretending they are available.
- Treats LSI as **multi-label classification** with `BCEWithLogitsLoss`, class imbalance handling, full development Macro-F1, and threshold selection.
- Handles long documents with two 512-token views (the beginning and end) and aggregates their representations.
- Runs a resumable experiment queue with distinct methods: adapters, LoRMA, partial/layer-wise fine-tuning, BitFit, full fine-tuning, and knowledge distillation.
- Saves resumable model/optimizer/scheduler/scaler/RNG state, best checkpoints, per-run submissions, metrics, and a portable resume bundle.

**Course contract:** the supplied specification requires all four Phase-1 model/architecture families, at least six substantially different experiments, compulsory LoRMA, and at least one knowledge-distillation experiment. Missing checkpoints are clearly marked; a missing architecture cannot be counted as completed.

**No score is guaranteed.** Choose the final model from the full development split, and do not use external pretrained models, external legal corpora, or external LLM APIs.

## Run
Attach your Phase-1 checkpoint datasets and select GPU. Choose **Run All**. If a run pauses at its session budget, save/download `LSI_PHASE5_RESUME_BUNDLE.zip`, attach it to the next session as an input, and Run All again. Tokenization progress is cached separately when the working directory survives.


In [ ]:
import os, sys, json, re, gc, math, time, random, hashlib, zipfile, shutil, warnings, ast, traceback, itertools
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
from tokenizers import Tokenizer
from IPython.display import display, FileLink

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
SEED=781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Python:",sys.version.split()[0],"| PyTorch:",torch.__version__,"| device:",DEVICE)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | VRAM={torch.cuda.get_device_properties(i).total_memory/1e9:.2f} GB")
else:
    print("WARNING: no GPU detected; Phase-5 training will be extremely slow on CPU.")
ROOT=Path("/kaggle/working/CS781_PHASE5_LSI")
DATA_DIR=ROOT/"data"; EXP_DIR=ROOT/"experiments"; SUB_DIR=ROOT/"submissions"; CACHE_DIR=ROOT/"token_cache_v1"
for d in [ROOT,DATA_DIR,EXP_DIR,SUB_DIR,CACHE_DIR]: d.mkdir(parents=True,exist_ok=True)
TOKENIZER_SHA256="3e38dd5c561d84ad8fbadf792bbca490f8c70346d922df7afbef37c5a942b71e"
OFFICIAL_REPO_ID="Exploration-Lab/CS-781-Capstone"
print("Output root:",ROOT)


In [ ]:

def read_hf_secret():
    token=os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token: return token
    try:
        from kaggle_secrets import UserSecretsClient
        client=UserSecretsClient()
        for name in ("HF","HF_TOKEN"):
            try:
                v=client.get_secret(name)
                if v and v.strip(): return v.strip()
            except Exception: pass
    except Exception: pass
    return None
HF_TOKEN=read_hf_secret()

_SKIP_DIRS={"token_cache_40k","token_cache","corpus_cache",".cache","__pycache__",".git","node_modules","lost+found"}
def iter_files_pruned(base, suffixes=None, names=None):
    base=Path(base)
    if not base.exists(): return
    suff={s.lower() for s in suffixes} if suffixes else None
    nms={s.lower() for s in names} if names else None
    for root,dirs,files in os.walk(base):
        dirs[:]=[d for d in dirs if d.lower() not in _SKIP_DIRS]
        for name in files:
            p=Path(root)/name
            if nms is not None and name.lower() not in nms: continue
            if suff is not None and p.suffix.lower() not in suff: continue
            yield p

def _safe_extract_resume_bundles():
    input_root=Path("/kaggle/input")
    for p in iter_files_pruned(input_root,suffixes={".zip"}):
        if "lsi_phase5_resume_bundle" not in p.name.lower(): continue
        try:
            print("Found portable resume bundle:",p,flush=True)
            with zipfile.ZipFile(p) as zf:
                for name in zf.namelist():
                    norm=Path(name)
                    if norm.is_absolute() or ".." in norm.parts: continue
                    if not (name.startswith("experiments/") or name.startswith("submissions/") or name in {"experiment_results.json","run_state.json","label_vocabulary.json","phase5_manifest.json"}): continue
                    dest=ROOT/name
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    if dest.exists() and dest.stat().st_size>0: continue
                    with zf.open(name) as src, open(dest,"wb") as out: shutil.copyfileobj(src,out)
            print("Resume bundle restored into",ROOT,flush=True)
        except Exception as e: print("Resume bundle warning:",p,repr(e)[:250],flush=True)
_safe_extract_resume_bundles()

# Phase-5 JSON schema: [{"id":...,"text":...,"labels":[integer statute IDs]}]
def _norm_label_ids(v):
    if v is None: return []
    if isinstance(v,dict):
        for k in ("labels","statute_ids","statutes","ids","label_ids","value"):
            if k in v: return _norm_label_ids(v[k])
        return list({int(k) for k,val in v.items() if bool(val) and str(k).lstrip("-+").isdigit()})
    if isinstance(v,(int,np.integer)): return [int(v)]
    if isinstance(v,float) and v.is_integer(): return [int(v)]
    if isinstance(v,str):
        v=v.strip()
        if not v: return []
        if v.startswith("["):
            try: return _norm_label_ids(json.loads(v))
            except Exception: pass
        pieces=re.split(r"[,;|\s]+",v)
        try: return [int(x) for x in pieces if x]
        except Exception: raise ValueError(f"Statute IDs must be integers; got {v[:100]!r}")
    if isinstance(v,(list,tuple,set,np.ndarray)):
        out=[]
        for x in v: out.extend(_norm_label_ids(x))
        return sorted(set(out))
    raise ValueError(f"Unsupported labels value type: {type(v).__name__}")

def normalize_lsi_records(raw,labeled):
    wrappers=("data","records","examples","items","train","dev","test","dataset","instances")
    for _ in range(6):
        if isinstance(raw,dict):
            key=next((k for k in wrappers if k in raw and isinstance(raw[k],(list,dict))),None)
            if key is not None: raw=raw[key]; continue
        break
    id_keys=("id","case_id","caseId","caseID","document_id","documentId","example_id","uid","case_no")
    text_keys=("text","document","content","case_text","judgment_text","judgement_text","full_text","judgment","judgement")
    label_keys=("labels","statute_ids","statutes","label_ids","applicable_statutes","relevant_statutes","target","y")
    if isinstance(raw,dict):
        rows=[]
        for k,v in raw.items():
            if not isinstance(v,dict): continue
            obj=dict(v); obj.setdefault("id",k); rows.append(obj)
    elif isinstance(raw,list): rows=raw
    else: return None
    out=[]
    for obj in rows:
        if not isinstance(obj,dict): return None
        lower={str(k).lower():k for k in obj}
        ik=next((lower[k.lower()] for k in id_keys if k.lower() in lower),None)
        tk=next((lower[k.lower()] for k in text_keys if k.lower() in lower and isinstance(obj[lower[k.lower()]],str)),None)
        lk=next((lower[k.lower()] for k in label_keys if k.lower() in lower),None) if labeled else None
        if ik is None or tk is None or (labeled and lk is None): return None
        rid=obj[ik]; text=obj[tk]
        if rid is None or not isinstance(text,str) or not text.strip(): return None
        row={"id":str(rid),"text":text}
        if labeled:
            labs=_norm_label_ids(obj[lk])
            row["labels"]=sorted(set(labs))
        out.append(row)
    if not out: return None
    # Merge exact duplicate training examples by unioning statute IDs; keep distinct texts separate.
    if labeled:
        merged={}
        for r in out:
            key=(r["id"],hashlib.sha1(r["text"].encode("utf-8","ignore")).hexdigest())
            if key not in merged: merged[key]=dict(r)
            else: merged[key]["labels"]=sorted(set(merged[key]["labels"])|set(r["labels"]))
        out=list(merged.values())
    return out

LSI_EXPECTED={"train":"Phase_5_LSI/Train/train.json","dev":"Phase_5_LSI/Train/dev.json","test":"Phase_5_LSI/Test/test.json"}
def discover_local_lsi():
    candidates={k:[] for k in LSI_EXPECTED}
    names={"train.json","dev.json","test.json"}
    for base in (Path("/kaggle/input"),DATA_DIR):
        for p in iter_files_pruned(base,suffixes={".json"},names=names):
            low=str(p).lower().replace("\\","/")
            if "/phase_5_lsi/" not in low and "/phase5_lsi/" not in low: continue
            if low.endswith("/phase_5_lsi/train/train.json") or low.endswith("/phase5_lsi/train/train.json"): candidates["train"].append(p)
            elif low.endswith("/phase_5_lsi/train/dev.json") or low.endswith("/phase5_lsi/train/dev.json"): candidates["dev"].append(p)
            elif low.endswith("/phase_5_lsi/test/test.json") or low.endswith("/phase5_lsi/test/test.json"): candidates["test"].append(p)
    return {k:sorted(v,key=lambda p:(len(str(p)),str(p)))[0] for k,v in candidates.items() if v}

def download_official_lsi(missing):
    from huggingface_hub import HfApi, hf_hub_download
    api=HfApi(token=HF_TOKEN)
    files=api.list_repo_files(repo_id=OFFICIAL_REPO_ID,repo_type="dataset")
    file_lows={f.lower():f for f in files}
    result={}
    for kind in missing:
        suffix=LSI_EXPECTED[kind].lower()
        matches=[f for f in files if f.lower().replace("\\","/").endswith(suffix)]
        if not matches:
            matches=[f for f in files if "phase_5_lsi" in f.lower() and f.lower().endswith("/"+kind+".json")]
        if not matches: raise FileNotFoundError(f"Official repository did not list a Phase-5 {kind}.json. Check dataset access and repository contents.")
        # Prefer the canonical exact path; never use a different phase's similarly named split.
        matches.sort(key=lambda f:(0 if f.lower().replace("\\","/")==suffix else 1,len(f),f))
        chosen=matches[0]
        local=hf_hub_download(repo_id=OFFICIAL_REPO_ID,filename=chosen,repo_type="dataset",token=HF_TOKEN)
        result[kind]=Path(local)
        print(f"Downloaded official Phase-5 {kind}: {chosen} -> {local}",flush=True)
    return result

DATA_PATHS=discover_local_lsi()
missing=[k for k in LSI_EXPECTED if k not in DATA_PATHS]
print("Local Phase-5 LSI candidates:",{k:str(v) for k,v in DATA_PATHS.items()},flush=True)
if missing:
    try: DATA_PATHS.update(download_official_lsi(missing))
    except Exception as e:
        raise RuntimeError("Could not find Phase_5_LSI/Train/train.json, Train/dev.json, and Test/test.json in attached inputs or download them from the official course dataset. Add Kaggle Secret HF/HF_TOKEN with dataset access and Internet if required. Original error: "+repr(e))

TRAIN_RECORDS=[]; DEV_RECORDS=[]; TEST_RECORDS=[]
def load_lsi_split(kind):
    p=DATA_PATHS[kind]
    with open(p,"r",encoding="utf-8") as f: raw=json.load(f)
    records=normalize_lsi_records(raw,labeled=(kind!="test"))
    if not records:
        if isinstance(raw,list) and raw and isinstance(raw[0],dict): print(kind,"sample fields:",list(raw[0])[:50])
        elif isinstance(raw,dict): print(kind,"top-level keys:",list(raw)[:30])
        raise RuntimeError(f"Could not normalize Phase-5 {kind} schema at {p}. The loader expects id, text, and labels (train/dev) or id/text (test); see printed field names.")
    ids=[r["id"] for r in records]
    if kind=="test" and len(ids)!=len(set(ids)): raise RuntimeError("Test IDs are not unique after normalization; refusing to create ambiguous predictions.")
    if kind!="test":
        label_hist=Counter(y for r in records for y in r["labels"])
        print(f"{kind.upper()}: {len(records):,} records | labels={len(label_hist):,} | min/median/max labels per row={min(map(lambda r:len(r['labels']),records))}/{float(np.median([len(r['labels']) for r in records]))}/{max(map(lambda r:len(r['labels']),records))}")
    else: print(f"TEST: {len(records):,} records | unique IDs={len(set(ids)):,}")
    return records
TRAIN_RECORDS=load_lsi_split("train"); DEV_RECORDS=load_lsi_split("dev"); TEST_RECORDS=load_lsi_split("test")

LABEL_IDS=sorted(set(y for r in TRAIN_RECORDS for y in r["labels"]) | set(y for r in DEV_RECORDS for y in r["labels"]))
LABEL_TO_COL={lab:i for i,lab in enumerate(LABEL_IDS)}
if not LABEL_IDS: raise RuntimeError("No statute IDs found in labeled Phase-5 training data.")
train_label_set=set(y for r in TRAIN_RECORDS for y in r["labels"])
dev_only=sorted(set(y for r in DEV_RECORDS for y in r["labels"])-train_label_set)
print("Statute vocabulary size:",len(LABEL_IDS),"| train-only labels:",len(train_label_set),"| dev labels unseen in train:",len(dev_only))
if dev_only: print("WARNING: development contains statute IDs absent from train; they cannot be learned from train examples:",dev_only[:25])
(ROOT/"label_vocabulary.json").write_text(json.dumps({str(i):int(lab) for i,lab in enumerate(LABEL_IDS)},indent=2),encoding="utf-8")

# Find checkpoint candidates quickly; skip token-cache trees and prefer best_model.pt over huge latest.pt files.
def _arch_hint(path):
    s=str(path).lower().replace("_","-").replace("\\","/")
    if any(x in s for x in ("legalprefixformer","prefixformer","arch-d","architecture-d","master-d")): return "D"
    if any(x in s for x in ("dmlc-150","dmlc150","c-arch","arch-c","architecture-c","master-c")): return "C"
    if any(x in s for x in ("legalhybrid150","arch-b","architecture-b","master-b")): return "B"
    if any(x in s for x in ("legaldecodera","arch-a","architecture-a","master-a","checkpoints_archa")): return "A"
    return None

def payload_state(payload):
    for k in ("model_state_dict","state_dict","model_weights","weights","model"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and v and any(torch.is_tensor(x) for x in v.values()): return v
    if isinstance(payload,dict) and payload and all(torch.is_tensor(v) for v in payload.values()): return payload
    return None

def payload_config(payload):
    for k in ("model_config","MODEL_CONFIG","config","cfg","model_cfg","model_config_dict"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and ("vocab_size" in v or "d_model" in v): return dict(v)
    return {}

CHECKPOINT_CANDIDATES={a:[] for a in "ABCD"}
for p in iter_files_pruned("/kaggle/input",suffixes={".pt",".pth",".ckpt"}):
    nm=p.name.lower()
    if nm not in {"best_model.pt","latest.pt","model.pt","checkpoint.pt","best.pt","checkpoint.pth","model.pth"}: continue
    a=_arch_hint(p)
    if a: CHECKPOINT_CANDIDATES[a].append(p)

def _cp_score(p,arch):
    s=str(p).lower().replace("\\","/")
    name=0 if p.name.lower()=="best_model.pt" else 1 if p.name.lower()=="model.pt" else 2 if p.name.lower()=="latest.pt" else 3
    strong={"A":["/arch-a/","checkpoints_archa_large"],"B":["arch-b-500m-traning","legalhybrid150"],"C":["c-arch","dmlc-150"],"D":["legalprefixformer","arch-d"]}[arch]
    priority=0 if any(k in s for k in strong) else 1
    return (name,priority,len(s),s)
for a in "ABCD":
    CHECKPOINT_CANDIDATES[a]=sorted(set(CHECKPOINT_CANDIDATES[a]),key=lambda p:_cp_score(p,a))[:6]
    print(f"Architecture {a} checkpoint candidates:",len(CHECKPOINT_CANDIDATES[a]))
    for p in CHECKPOINT_CANDIDATES[a][:3]: print(" ",p)

# Validate checkpoint metadata without deserializing more than a few candidates per architecture.
CHECKPOINT_PATHS={}; CHECKPOINT_META={}; ARCH_AVAILABLE=[]
for a in "ABCD":
    for p in CHECKPOINT_CANDIDATES[a]:
        payload=None
        try:
            try: payload=torch.load(p,map_location="cpu",weights_only=False,mmap=True)
            except (TypeError,RuntimeError,ValueError): payload=torch.load(p,map_location="cpu",weights_only=False)
            cfg=payload_config(payload); st=payload_state(payload)
            if not cfg or st is None:
                del payload
                print("  metadata/state missing; trying alternate:",p)
                continue
            inferred=_arch_hint(p)
            if inferred!=a:
                del payload; continue
            CHECKPOINT_PATHS[a]=str(p)
            CHECKPOINT_META[a]={"path":str(p),"cfg":cfg,"step":int(payload.get("step",payload.get("global_step",0)) or 0),"validation_loss":payload.get("validation_loss")}
            ARCH_AVAILABLE.append(a)
            print(f"  SELECTED {a}: {p} | d_model={cfg.get('d_model')} | step={CHECKPOINT_META[a]['step']}",flush=True)
            del payload,st
            break
        except Exception as e:
            print("  candidate metadata check failed; trying next:",p,type(e).__name__,str(e)[:200],flush=True)
            try: del payload
            except Exception: pass
if "A" not in ARCH_AVAILABLE and len(ARCH_AVAILABLE)==0:
    raise RuntimeError("No compatible Phase-1 A/B/C/D checkpoint found. Attach your existing Phase-1 checkpoint output datasets.")
MISSING_ARCHS=sorted(set("ABCD")-set(ARCH_AVAILABLE))
print("Compatible architectures:",ARCH_AVAILABLE,"| missing:",MISSING_ARCHS)

# Exact Phase-1 LEGAL_40K tokenizer fingerprint.
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""): h.update(block)
    return h.hexdigest()
TOK_CANDIDATES=[]
for p in iter_files_pruned("/kaggle/input",names={"tokenizer.json"}):
    if any(k in str(p).lower() for k in ("legal_bpe_40k","legal_40k","tokenizer")): TOK_CANDIDATES.append(p)
TOKENIZER_PATH=None
for p in sorted(set(TOK_CANDIDATES),key=lambda p:(0 if "legal_bpe_40k" in str(p).lower() else 1,len(str(p)))):
    try:
        if sha256_file(p)==TOKENIZER_SHA256: TOKENIZER_PATH=p; break
    except Exception: pass
if TOKENIZER_PATH is None:
    raise RuntimeError("The exact Phase-1 LEGAL_40K tokenizer was not found among attached inputs. Attach one of your Phase-1 output datasets containing tokenizers/legal_bpe_40k/tokenizer.json.")
TOKENIZER=Tokenizer.from_file(str(TOKENIZER_PATH))
assert TOKENIZER.get_vocab_size()==40000 and sha256_file(TOKENIZER_PATH)==TOKENIZER_SHA256
print("Tokenizer:",TOKENIZER_PATH,"| vocab:",TOKENIZER.get_vocab_size(),"| SHA256 verified")

DATA_SIGNATURE=hashlib.sha256((
    TOKENIZER_SHA256+"|"+str(len(TRAIN_RECORDS))+"|"+str(len(DEV_RECORDS))+"|"+str(len(TEST_RECORDS))+"|"+json.dumps(LABEL_IDS)+"|"+
    "|".join(hashlib.sha1((r['id']+'|'+str(len(r['text']))+'|'+','.join(map(str,r.get('labels',[])))).encode()).hexdigest() for r in TRAIN_RECORDS[:500])
).encode()).hexdigest()
(ROOT/"phase5_manifest.json").write_text(json.dumps({"repo_id":OFFICIAL_REPO_ID,"data_paths":{k:str(v) for k,v in DATA_PATHS.items()},"train_count":len(TRAIN_RECORDS),"dev_count":len(DEV_RECORDS),"test_count":len(TEST_RECORDS),"label_count":len(LABEL_IDS),"label_ids":LABEL_IDS,"checkpoint_paths":CHECKPOINT_PATHS,"missing_architectures":MISSING_ARCHS,"tokenizer_path":str(TOKENIZER_PATH),"tokenizer_sha256":TOKENIZER_SHA256,"data_signature":DATA_SIGNATURE},indent=2),encoding="utf-8")
print("PHASE-5 DATA AND CHECKPOINT DISCOVERY PASSED")


In [ ]:
# Namespaces are isolated because all four implementation files reuse names such as RMSNorm and SwiGLU.
ARCH_CLASS_NAMES = {"A":"LegalDecoderA", "B":"LegalHybrid150", "C":"DMLC150", "D":"LegalPrefixFormer145"}
ARCH_SOURCE = {}
ARCH_SOURCE['A'] = 'class RMSNorm(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        eps=1e-5\n    ):\n\n        super().__init__()\n\n        self.weight = nn.Parameter(\n            torch.ones(dim)\n        )\n\n        self.eps = eps\n\n    def forward(self, x):\n\n        inv_rms = torch.rsqrt(\n            x.pow(2).mean(\n                dim=-1,\n                keepdim=True\n            )\n            + self.eps\n        )\n\n        return (\n            x\n            * inv_rms\n            * self.weight\n        )\n\nclass RotaryEmbedding(nn.Module):\n\n    def __init__(\n        self,\n        dim,\n        max_seq_len,\n        theta=10000.0\n    ):\n\n        super().__init__()\n\n        if dim % 2:\n            raise ValueError(\n                "RoPE dimension must be even."\n            )\n\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(\n                    0,\n                    dim,\n                    2,\n                    dtype=torch.float32\n                )\n                / dim\n            )\n        )\n\n        positions = torch.arange(\n            max_seq_len,\n            dtype=torch.float32\n        )\n\n        freqs = torch.outer(\n            positions,\n            inv_freq\n        )\n\n        self.register_buffer(\n            "cos_cached",\n            torch.repeat_interleave(\n                freqs.cos(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n        self.register_buffer(\n            "sin_cached",\n            torch.repeat_interleave(\n                freqs.sin(),\n                2,\n                dim=-1\n            ),\n            persistent=False\n        )\n\n    def forward(\n        self,\n        x,\n        position_offset=0\n    ):\n\n        seq_len = x.size(-2)\n\n        end = (\n            position_offset\n            + seq_len\n        )\n\n        if (\n            end\n            > self.cos_cached.size(0)\n        ):\n            raise ValueError(\n                "RoPE context exceeded."\n            )\n\n        cos = self.cos_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        sin = self.sin_cached[\n            position_offset:end\n        ].to(\n            device=x.device,\n            dtype=x.dtype\n        )\n\n        cos = cos.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        sin = sin.view(\n            1,\n            1,\n            seq_len,\n            -1\n        )\n\n        even = x[..., 0::2]\n        odd = x[..., 1::2]\n\n        rotated = torch.stack(\n            [\n                -odd,\n                even\n            ],\n            dim=-1\n        ).flatten(-2)\n\n        return (\n            x * cos\n            + rotated * sin\n        )\n\nclass GQAAttention(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        n_heads = cfg[\n            "n_heads"\n        ]\n\n        n_kv_heads = cfg[\n            "n_kv_heads"\n        ]\n\n        head_dim = (\n            d_model // n_heads\n        )\n\n        self.n_heads = n_heads\n        self.n_kv_heads = n_kv_heads\n        self.head_dim = head_dim\n\n        self.group_size = (\n            n_heads\n            // n_kv_heads\n        )\n\n        self.q_proj = nn.Linear(\n            d_model,\n            n_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.k_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.v_proj = nn.Linear(\n            d_model,\n            n_kv_heads * head_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.o_proj = nn.Linear(\n            d_model,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n        self.rope = RotaryEmbedding(\n            head_dim,\n            cfg["context_length"],\n            cfg["rope_theta"]\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        batch, seq_len, d_model = (\n            x.shape\n        )\n\n        past_len = (\n            0\n            if past_kv is None\n            else past_kv[0].size(2)\n        )\n\n        q = self.q_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        k = self.k_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        v = self.v_proj(\n            x\n        ).view(\n            batch,\n            seq_len,\n            self.n_kv_heads,\n            self.head_dim\n        ).transpose(\n            1,\n            2\n        )\n\n        q = self.rope(\n            q,\n            past_len\n        )\n\n        k = self.rope(\n            k,\n            past_len\n        )\n\n        if past_kv is not None:\n\n            past_k, past_v = past_kv\n\n            k = torch.cat(\n                [past_k, k],\n                dim=2\n            )\n\n            v = torch.cat(\n                [past_v, v],\n                dim=2\n            )\n\n        if (\n            self.n_kv_heads\n            != self.n_heads\n        ):\n\n            k_attn = (\n                k.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n            v_attn = (\n                v.repeat_interleave(\n                    self.group_size,\n                    dim=1\n                )\n            )\n\n        else:\n\n            k_attn = k\n            v_attn = v\n\n        if past_kv is None:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=True,\n            )\n\n        else:\n\n            y = F.scaled_dot_product_attention(\n                q,\n                k_attn,\n                v_attn,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n        y = y.transpose(\n            1,\n            2\n        ).contiguous().view(\n            batch,\n            seq_len,\n            d_model\n        )\n\n        output = self.o_proj(\n            y\n        )\n\n        if use_cache:\n            return output, (k, v)\n\n        return output\n\nclass SwiGLU(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        ffn_dim = cfg[\n            "ffn_dim"\n        ]\n\n        self.gate_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.up_proj = nn.Linear(\n            d_model,\n            ffn_dim,\n            bias=cfg["use_bias"]\n        )\n\n        self.down_proj = nn.Linear(\n            ffn_dim,\n            d_model,\n            bias=cfg["use_bias"]\n        )\n\n    def forward(self, x):\n\n        return self.down_proj(\n            F.silu(\n                self.gate_proj(x)\n            )\n            * self.up_proj(x)\n        )\n\nclass DecoderBlock(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        d_model = cfg[\n            "d_model"\n        ]\n\n        self.norm1 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.attn = GQAAttention(\n            cfg\n        )\n\n        self.norm2 = RMSNorm(\n            d_model,\n            cfg["norm_eps"]\n        )\n\n        self.ffn = SwiGLU(\n            cfg\n        )\n\n    def forward(\n        self,\n        x,\n        past_kv=None,\n        use_cache=False\n    ):\n\n        if use_cache:\n\n            attn_out, present = (\n                self.attn(\n                    self.norm1(x),\n                    past_kv=past_kv,\n                    use_cache=True,\n                )\n            )\n\n            x = x + attn_out\n\n            x = x + self.ffn(\n                self.norm2(x)\n            )\n\n            return x, present\n\n        x = x + self.attn(\n            self.norm1(x)\n        )\n\n        x = x + self.ffn(\n            self.norm2(x)\n        )\n\n        return x\n\nclass LegalDecoderA(nn.Module):\n\n    def __init__(\n        self,\n        cfg\n    ):\n\n        super().__init__()\n\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"],\n            cfg["d_model"]\n        )\n\n        self.blocks = nn.ModuleList([\n            DecoderBlock(cfg)\n            for _ in range(\n                cfg["n_layers"]\n            )\n        ])\n\n        self.final_norm = RMSNorm(\n            cfg["d_model"],\n            cfg["norm_eps"]\n        )\n\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False\n        )\n\n        if cfg["tie_embeddings"]:\n\n            self.lm_head.weight = (\n                self.token_embedding.weight\n            )\n\n        self.apply(\n            self._init_weights\n        )\n\n        residual_scale = (\n            1.0\n            / math.sqrt(\n                2.0\n                * cfg["n_layers"]\n            )\n        )\n\n        for block in self.blocks:\n\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02\n                * residual_scale,\n            )\n\n    def _init_weights(\n        self,\n        module\n    ):\n\n        if isinstance(\n            module,\n            nn.Linear\n        ):\n\n            if module.bias is not None:\n                nn.init.zeros_(\n                    module.bias\n                )\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02,\n            )\n\n        elif isinstance(\n            module,\n            nn.Embedding\n        ):\n\n            nn.init.normal_(\n                module.weight,\n                mean=0.0,\n                std=0.02\n            )\n\n        elif isinstance(\n            module,\n            RMSNorm\n        ):\n\n            nn.init.ones_(\n                module.weight\n            )\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None\n    ):\n\n        x = self.token_embedding(\n            input_ids\n        )\n\n        if use_cache:\n\n            if past_key_values is None:\n\n                past_key_values = [\n                    None\n                    for _ in self.blocks\n                ]\n\n            presents = []\n\n            for block, past in zip(\n                self.blocks,\n                past_key_values\n            ):\n\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True\n                )\n\n                presents.append(\n                    present\n                )\n\n            x = self.final_norm(\n                x\n            )\n\n            logits = self.lm_head(\n                x\n            )\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n\n            if (\n                self.training\n                and self.cfg[\n                    "gradient_checkpointing"\n                ]\n            ):\n\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False\n                )\n\n            else:\n\n                x = block(x)\n\n        x = self.final_norm(\n            x\n        )\n\n        logits = self.lm_head(\n            x\n        )\n\n        loss = None\n\n        if labels is not None:\n\n            loss = F.cross_entropy(\n                logits.reshape(\n                    -1,\n                    logits.size(-1)\n                ),\n                labels.reshape(\n                    -1\n                )\n            )\n\n        return {\n            "logits": logits,\n            "loss": loss,\n        }'
ARCH_SOURCE['B'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        # Stable fp32 RMS computation, then explicitly restore the incoming dtype.\n        inv_rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        )\n        scale = self.weight.to(dtype=x.dtype)\n        out = (x * inv_rms.to(dtype=x.dtype)) * scale\n        return out.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, dim, 2, dtype=torch.float32) / dim\n            )\n        )\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def forward(self, x, start_pos=0):\n        seq_len = x.size(-2)\n        end = start_pos + seq_len\n        if end > self.cos_cached.size(0):\n            raise RuntimeError(\n                f"RoPE request {end} exceeds max context {self.cos_cached.size(0)}"\n            )\n        cos = self.cos_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        sin = self.sin_cached[start_pos:end].to(dtype=x.dtype)[None, None, :, :]\n        x1 = x[..., :x.size(-1)//2]\n        x2 = x[..., x.size(-1)//2:]\n        rotated = torch.cat([-x2, x1], dim=-1)\n        return x * cos + rotated * sin\n\nclass HybridGQAAttention(nn.Module):\n    def __init__(self, cfg, is_global):\n        super().__init__()\n        d = cfg["d_model"]\n        h = cfg["n_heads"]\n        kvh = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        if d != h * hd:\n            raise ValueError("d_model must equal n_heads * head_dim.")\n        if h % kvh != 0:\n            raise ValueError("n_heads must be divisible by n_kv_heads.")\n\n        self.d_model = d\n        self.n_heads = h\n        self.n_kv_heads = kvh\n        self.head_dim = hd\n        self.group_size = h // kvh\n\n        self.is_global = bool(is_global)\n        self.local_window = int(cfg["local_window"])\n        self.query_chunk = int(cfg["local_query_chunk"])\n\n        self.q_proj = nn.Linear(d, h * hd, bias=cfg["use_bias"])\n        self.k_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.v_proj = nn.Linear(d, kvh * hd, bias=cfg["use_bias"])\n        self.o_proj = nn.Linear(d, d, bias=cfg["use_bias"])\n\n        self.rope = RotaryEmbedding(\n            hd,\n            cfg["context_length"],\n            cfg["rope_theta"],\n        )\n\n    def _expand_kv(self, k, v):\n        if self.n_kv_heads == self.n_heads:\n            return k, v\n        k = k.repeat_interleave(self.group_size, dim=1)\n        v = v.repeat_interleave(self.group_size, dim=1)\n        return k, v\n\n    def _local_attention_train(self, q, k_full, v_full, past_len):\n        q_len = q.size(-2)\n        outputs = []\n\n        for start in range(0, q_len, self.query_chunk):\n            end = min(q_len, start + self.query_chunk)\n            curr = end - start\n\n            abs_q_start = past_len + start\n            key_start_abs = max(0, abs_q_start - self.local_window + 1)\n            key_end_abs = past_len + end\n\n            k_slice = k_full[:, :, key_start_abs:key_end_abs, :]\n            v_slice = v_full[:, :, key_start_abs:key_end_abs, :]\n\n            k_len = k_slice.size(-2)\n\n            # Exact causal sliding-window mask for this chunk. Construct only\n            # [query_chunk, local_window] rather than a full [T,T] mask.\n            q_positions = torch.arange(\n                abs_q_start,\n                past_len + end,\n                device=q.device,\n                dtype=torch.long,\n            )[:, None]\n            k_positions = torch.arange(\n                key_start_abs,\n                key_end_abs,\n                device=q.device,\n                dtype=torch.long,\n            )[None, :]\n            mask = (\n                (k_positions <= q_positions)\n                & (k_positions >= (q_positions - self.local_window + 1))\n            )\n            assert mask.shape == (curr, k_len)\n\n            y = F.scaled_dot_product_attention(\n                q[:, :, start:end, :],\n                k_slice,\n                v_slice,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n            outputs.append(y)\n\n        return torch.cat(outputs, dim=-2)\n\n    def _local_attention_cached(self, q, k_full, v_full):\n        # Intended generation path: q_len=1. All retained keys are historical\n        # tokens plus the current token, so no causal mask is needed.\n        keep = min(self.local_window, k_full.size(-2))\n        k_slice = k_full[:, :, -keep:, :]\n        v_slice = v_full[:, :, -keep:, :]\n        return F.scaled_dot_product_attention(\n            q,\n            k_slice,\n            v_slice,\n            attn_mask=None,\n            dropout_p=0.0,\n            is_causal=False,\n        )\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        bsz, q_len, _ = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        if past_len + q_len > self.rope.cos_cached.size(0):\n            raise RuntimeError("Attention request exceeds context length.")\n\n        q = self.q_proj(x).view(\n            bsz, q_len, self.n_heads, self.head_dim\n        ).transpose(1, 2)\n\n        k = self.k_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        v = self.v_proj(x).view(\n            bsz, q_len, self.n_kv_heads, self.head_dim\n        ).transpose(1, 2)\n\n        q = self.rope(q, past_len)\n        k = self.rope(k, past_len)\n\n        if past_kv is not None:\n            past_k, past_v = past_kv\n            k_full = torch.cat([past_k, k], dim=2)\n            v_full = torch.cat([past_v, v], dim=2)\n        else:\n            k_full = k\n            v_full = v\n\n        if self.n_kv_heads != self.n_heads:\n            k_attn, v_attn = self._expand_kv(k_full, v_full)\n        else:\n            k_attn, v_attn = k_full, v_full\n\n        if self.is_global:\n            if past_kv is None:\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=True,\n                )\n            else:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached global attention currently expects q_len=1."\n                    )\n                y = F.scaled_dot_product_attention(\n                    q,\n                    k_attn,\n                    v_attn,\n                    attn_mask=None,\n                    dropout_p=0.0,\n                    is_causal=False,\n                )\n        else:\n            if past_kv is not None:\n                if q_len != 1:\n                    raise RuntimeError(\n                        "Cached local attention currently expects q_len=1."\n                    )\n                y = self._local_attention_cached(q, k_attn, v_attn)\n            else:\n                y = self._local_attention_train(\n                    q, k_attn, v_attn, past_len=0\n                )\n\n        y = y.transpose(1, 2).contiguous().view(\n            bsz, q_len, self.d_model\n        )\n        y = self.o_proj(y)\n\n        if use_cache:\n            return y, (k_full, v_full)\n        return y\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        f = cfg["ffn_dim"]\n        self.gate_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.up_proj = nn.Linear(d, f, bias=cfg["use_bias"])\n        self.down_proj = nn.Linear(f, d, bias=cfg["use_bias"])\n\n    def forward(self, x):\n        return self.down_proj(\n            F.silu(self.gate_proj(x)) * self.up_proj(x)\n        )\n\nclass HybridBlock(nn.Module):\n    def __init__(self, cfg, layer_index):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = HybridGQAAttention(\n            cfg,\n            is_global=((layer_index + 1) % cfg["global_every"] == 0),\n        )\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, past_kv=None, use_cache=False):\n        if use_cache:\n            attn_out, present = self.attn(\n                self.norm1(x),\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + attn_out\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x))\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalHybrid150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n\n        self.token_embedding = nn.Embedding(\n            cfg["vocab_size"], cfg["d_model"]\n        )\n        self.blocks = nn.ModuleList([\n            HybridBlock(cfg, i)\n            for i in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(\n            cfg["d_model"], cfg["norm_eps"]\n        )\n        self.lm_head = nn.Linear(\n            cfg["d_model"],\n            cfg["vocab_size"],\n            bias=False,\n        )\n\n        # Initialize parameters once, then tie the LM head.\n        # This avoids double-initializing the same shared Parameter.\n        self.apply(self._init_weights)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        elif isinstance(module, RMSNorm):\n            nn.init.ones_(module.weight)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        use_cache=False,\n        past_key_values=None,\n    ):\n        input_ids = input_ids.long()\n        x = self.token_embedding(input_ids)\n        # Keep the residual stream in FP16 on CUDA under the training/inference\n        # autocast path; this avoids unintended FP32 residual accumulation.\n        if x.is_cuda and x.dtype != torch.float16:\n            x = x.to(dtype=torch.float16)\n\n        if use_cache:\n            if past_key_values is None:\n                past_key_values = [None] * len(self.blocks)\n\n            presents = []\n            for block, past in zip(self.blocks, past_key_values):\n                x, present = block(\n                    x,\n                    past_kv=past,\n                    use_cache=True,\n                )\n                presents.append(present)\n\n            x = self.final_norm(x)\n            logits = self.lm_head(x)\n\n            return {\n                "logits": logits,\n                "loss": None,\n                "past_key_values": presents,\n            }\n\n        for block in self.blocks:\n            if (\n                self.training\n                and self.cfg["gradient_checkpointing"]\n            ):\n                x = torch.utils.checkpoint.checkpoint(\n                    block,\n                    x,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x)\n\n        x = self.final_norm(x)\n\n        if labels is not None:\n            # Keep peak memory reasonable by projecting in time chunks.\n            labels = labels.long()\n            time_chunk = 512\n            losses = []\n            for start in range(0, x.size(1), time_chunk):\n                end = min(x.size(1), start + time_chunk)\n                logits_chunk = self.lm_head(x[:, start:end, :])\n                loss_chunk = F.cross_entropy(\n                    logits_chunk.reshape(-1, logits_chunk.size(-1)),\n                    labels[:, start:end].reshape(-1),\n                )\n                losses.append(loss_chunk)\n                del logits_chunk\n            loss = torch.stack(losses).mean()\n            return {"loss": loss}\n\n        logits = self.lm_head(x)\n        return {"logits": logits, "loss": None}\n\ndef estimate_model_params(cfg):\n    v = cfg["vocab_size"]\n    d = cfg["d_model"]\n    L = cfg["n_layers"]\n    H = cfg["n_heads"]\n    KV = cfg["n_kv_heads"]\n    hd = cfg["head_dim"]\n    f = cfg["ffn_dim"]\n\n    embedding = v * d\n    attention = (\n        d * H * hd\n        + d * KV * hd\n        + d * KV * hd\n        + (H * hd) * d\n    )\n    ffn = 3 * d * f\n    norms = (2 * L + 1) * d\n    output_head = 0 if cfg["tie_embeddings"] else v * d\n\n    return embedding + L * (attention + ffn) + norms + output_head'
ARCH_SOURCE['C'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        inv_rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)\n        return x * inv_rms.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, dim, max_seq_len, theta=10_000.0):\n        super().__init__()\n        if dim % 2:\n            raise ValueError("RoPE dimension must be even.")\n        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))\n        positions = torch.arange(max_seq_len, dtype=torch.float32)\n        freqs = torch.outer(positions, inv_freq)\n        emb = torch.cat([freqs, freqs], dim=-1)\n        self.register_buffer("cos_cached", emb.cos(), persistent=False)\n        self.register_buffer("sin_cached", emb.sin(), persistent=False)\n\n    def get(self, start, length, device, dtype):\n        if start + length > self.cos_cached.size(0):\n            raise RuntimeError("Requested RoPE positions exceed context length.")\n        cos = self.cos_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        sin = self.sin_cached[start:start+length].to(device=device, dtype=dtype).view(1, 1, length, -1)\n        return cos, sin\n\ndef rotate_half(x):\n    d = x.size(-1)\n    return torch.cat((-x[..., d//2:], x[..., :d//2]), dim=-1)\n\ndef apply_rope(x, cos, sin):\n    return x * cos + rotate_half(x) * sin\n\nclass GQAAttention(nn.Module):\n    def __init__(self, d_model, q_heads, kv_heads, head_dim, max_seq_len, theta=10_000.0, causal=False, use_rope=True):\n        super().__init__()\n        if d_model != q_heads * head_dim:\n            raise ValueError("d_model must equal q_heads * head_dim")\n        if q_heads % kv_heads:\n            raise ValueError("q_heads must be divisible by kv_heads")\n        self.q_heads = q_heads\n        self.kv_heads = kv_heads\n        self.head_dim = head_dim\n        self.causal = causal\n        self.use_rope = use_rope\n        self.q_proj = nn.Linear(d_model, q_heads * head_dim, bias=False)\n        self.k_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.v_proj = nn.Linear(d_model, kv_heads * head_dim, bias=False)\n        self.o_proj = nn.Linear(q_heads * head_dim, d_model, bias=False)\n        self.rope = RotaryEmbedding(head_dim, max_seq_len, theta) if use_rope else None\n\n    def _q(self, x):\n        return self.q_proj(x).view(x.size(0), x.size(1), self.q_heads, self.head_dim).transpose(1, 2)\n\n    def _kv(self, x):\n        k = self.k_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        v = self.v_proj(x).view(x.size(0), x.size(1), self.kv_heads, self.head_dim).transpose(1, 2)\n        return k, v\n\n    def _repeat_kv(self, x):\n        return x.repeat_interleave(self.q_heads // self.kv_heads, dim=1)\n\n    @staticmethod\n    def _mask(mask):\n        if mask is None:\n            return None\n        if mask.dim() == 2:\n            return mask[:, None, None, :]\n        if mask.dim() == 3:\n            return mask[:, None, :, :]\n        return mask\n\n    def forward(self, q_input, kv_input=None, attn_mask=None, cache=None, use_cache=False, position_offset=0):\n        is_cross = kv_input is not None\n        source = q_input if kv_input is None else kv_input\n        q = self._q(q_input)\n        k_new, v_new = self._kv(source)\n\n        if self.use_rope and not is_cross:\n            cos_k, sin_k = self.rope.get(position_offset, k_new.size(2), k_new.device, k_new.dtype)\n            cos_q, sin_q = self.rope.get(position_offset, q.size(2), q.device, q.dtype)\n            k_new = apply_rope(k_new, cos_k, sin_k)\n            q = apply_rope(q, cos_q, sin_q)\n\n        if cache is not None:\n            k_cache = torch.cat([cache[0], k_new], dim=2)\n            v_cache = torch.cat([cache[1], v_new], dim=2)\n        else:\n            k_cache, v_cache = k_new, v_new\n\n        k = self._repeat_kv(k_cache)\n        v = self._repeat_kv(v_cache)\n        mask = self._mask(attn_mask)\n\n        if self.causal and cache is None and mask is None:\n            out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)\n        else:\n            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)\n\n        out = out.transpose(1, 2).contiguous().view(q_input.size(0), q_input.size(1), -1)\n        out = self.o_proj(out)\n        new_cache = (k_cache, v_cache) if use_cache else None\n        return out, new_cache\n\nclass SwiGLU(nn.Module):\n    def __init__(self, d_model, ffn_dim):\n        super().__init__()\n        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)\n\n    def forward(self, x):\n        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))\n\nclass EncoderBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["encoder_heads"], cfg["encoder_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=True)\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward(self, x, attention_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=attention_mask)[0]\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass DecoderBlock(nn.Module):\n    def __init__(self, cfg, fusion=False):\n        super().__init__()\n        d = cfg["d_model"]\n        self.fusion = bool(fusion)\n        self.norm1 = RMSNorm(d, cfg["norm_eps"])\n        self.self_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=True, use_rope=True)\n        if self.fusion:\n            self.cross_norm = RMSNorm(d, cfg["norm_eps"])\n            self.case_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            self.evidence_cross_attn = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n            gh = cfg["gate_hidden_dim"]\n            self.gate = nn.Sequential(nn.Linear(d, gh, bias=True), nn.Linear(gh, 1, bias=True))\n        self.norm2 = RMSNorm(d, cfg["norm_eps"])\n        self.ffn = SwiGLU(d, cfg["ffn_dim"])\n\n    def forward_full(self, x, evidence, case, evidence_mask, decoder_mask):\n        x = x + self.self_attn(self.norm1(x), attn_mask=decoder_mask)[0]\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x\n\n    def forward_step(self, x, evidence, case, evidence_mask, cache, position):\n        y, new_cache = self.self_attn(self.norm1(x), cache=cache, use_cache=True, position_offset=position)\n        x = x + y\n        if self.fusion:\n            u = self.cross_norm(x)\n            case_out = self.case_cross_attn(u, case)[0]\n            evidence_out = self.evidence_cross_attn(u, evidence, attn_mask=evidence_mask)[0]\n            gate = torch.sigmoid(self.gate(u))\n            x = x + gate * evidence_out + (1.0 - gate) * case_out\n        x = x + self.ffn(self.norm2(x))\n        return x, new_cache\n\nclass DMLC150(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = int(cfg["d_model"])\n        V = int(cfg["vocab_size"])\n        self.token_embedding = nn.Embedding(V, d)\n        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["encoder_layers"])])\n        self.encoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.case_memory = nn.Parameter(torch.empty(cfg["case_memory_slots"], d))\n        self.case_memory_slots = int(cfg["case_memory_slots"])\n        self.case_memory_attention = GQAAttention(d, cfg["decoder_q_heads"], cfg["decoder_kv_heads"], cfg["head_dim"], cfg["context_length"], cfg["rope_theta"], causal=False, use_rope=False)\n        self.case_memory_norm = RMSNorm(d, cfg["norm_eps"])\n        fusion = set(int(x) for x in cfg["memory_fusion_layers"])\n        self.decoder = nn.ModuleList([DecoderBlock(cfg, i in fusion) for i in range(1, cfg["decoder_layers"] + 1)])\n        self.fusion_layers = nn.ModuleDict({str(i): self.decoder[i - 1] for i in sorted(fusion)})\n        self.decoder_final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.decoder_start = nn.Parameter(torch.empty(d))\n        self.lm_head = nn.Linear(d, V, bias=False)\n        self.lm_head.weight = self.token_embedding.weight\n        self._reset_parameters(cfg)\n\n    def _reset_parameters(self, cfg):\n        total_residual_layers = cfg["encoder_layers"] + cfg["decoder_layers"]\n        small_std = 0.02 / math.sqrt(max(1.0, 2.0 * total_residual_layers))\n        for name, module in self.named_modules():\n            if isinstance(module, nn.Linear):\n                use_small = name.endswith("o_proj") or name.endswith("down_proj")\n                std = small_std if use_small else 0.02\n                nn.init.normal_(module.weight, mean=0.0, std=std)\n                if module.bias is not None:\n                    nn.init.zeros_(module.bias)\n            elif isinstance(module, nn.Embedding):\n                nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            elif isinstance(module, RMSNorm):\n                nn.init.ones_(module.weight)\n        nn.init.normal_(self.case_memory, mean=0.0, std=0.02)\n        nn.init.normal_(self.decoder_start, mean=0.0, std=0.02)\n        # Start with an unbiased 50/50 evidence-vs-case mixture.\n        for layer in self.fusion_layers.values():\n            nn.init.zeros_(layer.gate[-1].weight)\n            nn.init.zeros_(layer.gate[-1].bias)\n\n    def _encode(self, src_ids, src_attention_mask):\n        x = self.token_embedding(src_ids)\n        for layer in self.encoder:\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer, x, src_attention_mask, use_reentrant=False)\n            else:\n                x = layer(x, src_attention_mask)\n        evidence = self.encoder_final_norm(x)\n        case_seed = self.case_memory.unsqueeze(0).expand(src_ids.size(0), -1, -1)\n        case = case_seed + self.case_memory_attention(case_seed, evidence, attn_mask=src_attention_mask)[0]\n        case = self.case_memory_norm(case)\n        return evidence, case\n\n    def encode_prompt(self, src_ids, src_attention_mask=None):\n        if src_ids.dim() != 2:\n            raise ValueError("src_ids must be [B,S].")\n        if src_ids.size(1) > self.cfg["context_length"]:\n            raise RuntimeError("Source sequence exceeds context length.")\n        evidence, case = self._encode(src_ids, src_attention_mask)\n        return {"evidence_memory": evidence, "case_memory": case, "src_attention_mask": src_attention_mask}\n\n    def decode_teacher_forced(self, decoder_input_ids, memory, decoder_attention_mask=None):\n        if decoder_input_ids.dim() != 2:\n            raise ValueError("decoder_input_ids must be [B,T].")\n        B, T = decoder_input_ids.shape\n        if T > self.cfg["context_length"]:\n            raise RuntimeError("Target sequence exceeds decoder context length.")\n        start = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        shifted = self.token_embedding(decoder_input_ids[:, :-1]) if T > 1 else decoder_input_ids.new_empty((B, 0, self.cfg["d_model"]), dtype=torch.float32)\n        x = torch.cat([start, shifted], dim=1)\n        T2 = x.size(1)\n        if decoder_attention_mask is None:\n            causal = None  # Let SDPA use its fused causal path.\n        else:\n            causal = torch.tril(torch.ones((T2, T2), dtype=torch.bool, device=x.device))[None, None]\n            causal = causal & decoder_attention_mask[:, None, None, :]\n        evidence = memory["evidence_memory"]\n        case = memory["case_memory"]\n        evidence_mask = memory["src_attention_mask"]\n        for i, layer in enumerate(self.decoder, start=1):\n            if self.training and self.cfg.get("gradient_checkpointing", True):\n                x = checkpoint(layer.forward_full, x, evidence, case, evidence_mask, causal, use_reentrant=False)\n            else:\n                x = layer.forward_full(x, evidence, case, evidence_mask, causal)\n        return self.decoder_final_norm(x)\n\n    def forward(self, src_ids, decoder_input_ids=None, labels=None, src_attention_mask=None, decoder_attention_mask=None, return_logits=True):\n        if labels is not None and decoder_input_ids is None:\n            decoder_input_ids = labels\n        if decoder_input_ids is None:\n            raise ValueError("Provide decoder_input_ids or labels.")\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        hidden = self.decode_teacher_forced(decoder_input_ids, memory, decoder_attention_mask)\n        logits = F.linear(hidden, self.token_embedding.weight)\n        out = {}\n        if labels is not None:\n            out["loss"] = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)\n        if return_logits:\n            out["logits"] = logits\n        return out\n\n    @torch.no_grad()\n    def generate(self, src_ids, src_attention_mask=None, max_new_tokens=512, temperature=0.10, top_p=0.95, top_k=20, special_token_ids=None):\n        if special_token_ids is None:\n            special_token_ids = set()\n        memory = self.encode_prompt(src_ids, src_attention_mask)\n        B = src_ids.size(0)\n        x = self.decoder_start.view(1, 1, -1).expand(B, 1, -1)\n        caches = [None] * len(self.decoder)\n        generated = []\n        for position in range(int(max_new_tokens)):\n            for i, layer in enumerate(self.decoder):\n                x, caches[i] = layer.forward_step(x, memory["evidence_memory"], memory["case_memory"], memory["src_attention_mask"], caches[i], position)\n            logits = F.linear(self.decoder_final_norm(x)[:, -1], self.token_embedding.weight).float()\n            for token_id in special_token_ids:\n                if 0 <= int(token_id) < logits.size(-1):\n                    logits[:, int(token_id)] = -float("inf")\n            if temperature is None or float(temperature) <= 0:\n                next_ids = torch.argmax(logits, dim=-1)\n            else:\n                logits = logits / max(float(temperature), 1e-5)\n                if top_k and 0 < int(top_k) < logits.size(-1):\n                    values, _ = torch.topk(logits, int(top_k), dim=-1)\n                    cutoff = values[:, -1, None]\n                    logits = torch.where(logits < cutoff, torch.full_like(logits, -float("inf")), logits)\n                if top_p is not None and 0 < float(top_p) < 1:\n                    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)\n                    probs = torch.softmax(sorted_logits, dim=-1)\n                    cumulative = torch.cumsum(probs, dim=-1)\n                    remove = cumulative > float(top_p)\n                    remove[:, 1:] = remove[:, :-1].clone()\n                    remove[:, 0] = False\n                    sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))\n                    logits = torch.full_like(logits, -float("inf")).scatter(1, sorted_indices, sorted_logits)\n                next_ids = torch.multinomial(torch.softmax(logits, dim=-1), 1).squeeze(-1)\n            generated.append(next_ids)\n            x = self.token_embedding(next_ids).unsqueeze(1)\n        return torch.stack(generated, dim=1) if generated else src_ids.new_empty((B, 0))'
ARCH_SOURCE['D'] = 'class RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-5):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, x):\n        rms = torch.rsqrt(\n            x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps\n        ).to(x.dtype)\n        return x * rms * self.weight.to(x.dtype)\n\nclass RoPE(nn.Module):\n    def __init__(self, head_dim, max_len, theta=10000.0):\n        super().__init__()\n        inv_freq = 1.0 / (\n            theta ** (\n                torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim\n            )\n        )\n        pos = torch.arange(max_len, dtype=torch.float32)\n        freqs = torch.outer(pos, inv_freq)\n        self.register_buffer("cos", freqs.cos(), persistent=False)\n        self.register_buffer("sin", freqs.sin(), persistent=False)\n\n    def rotate(self, x, offset=0):\n        t = x.size(-2)\n        c = self.cos[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        s = self.sin[offset:offset+t].to(x.device, x.dtype)[None, None, :, :]\n        xe, xo = x[..., 0::2], x[..., 1::2]\n        y = torch.empty_like(x)\n        y[..., 0::2] = xe * c - xo * s\n        y[..., 1::2] = xe * s + xo * c\n        return y\n\nclass PrefixGQAAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d = cfg["d_model"]\n        H = cfg["n_heads"]\n        KV = cfg["n_kv_heads"]\n        hd = cfg["head_dim"]\n\n        self.H = H\n        self.KV = KV\n        self.hd = hd\n        self.group = H // KV\n\n        self.q_proj = nn.Linear(d, H * hd, bias=False)\n        self.k_proj = nn.Linear(d, KV * hd, bias=False)\n        self.v_proj = nn.Linear(d, KV * hd, bias=False)\n        self.o_proj = nn.Linear(d, d, bias=False)\n\n        self.rope = RoPE(hd, cfg["context_length"], cfg["rope_theta"])\n        self.q_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n        self.k_norm = RMSNorm(hd, cfg["norm_eps"]) if cfg.get("qk_norm") else nn.Identity()\n\n        suffix_len = cfg["context_length"] - cfg["prefix_length"]\n        prefix_len = cfg["prefix_length"]\n\n        # Small suffix mask: [suffix_queries, prefix+suffix_keys].\n        mask = torch.ones(\n            suffix_len, prefix_len + suffix_len, dtype=torch.bool\n        )\n        mask[:, prefix_len:] = torch.tril(\n            torch.ones(suffix_len, suffix_len, dtype=torch.bool)\n        )\n        self.register_buffer(\n            "suffix_mask",\n            mask[None, None, :, :],\n            persistent=False,\n        )\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        B, T, D = x.shape\n        past_len = 0 if past_kv is None else past_kv[0].size(2)\n\n        q = self.q_proj(x).view(B, T, self.H, self.hd).transpose(1, 2)\n        k = self.k_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n        v = self.v_proj(x).view(B, T, self.KV, self.hd).transpose(1, 2)\n\n        q = self.q_norm(self.rope.rotate(q, past_len))\n        k = self.k_norm(self.rope.rotate(k, past_len))\n\n        if past_kv is not None:\n            pk, pv = past_kv\n            k = torch.cat([pk, k], dim=2)\n            v = torch.cat([pv, v], dim=2)\n\n        ka = k.repeat_interleave(self.group, dim=1)\n        va = v.repeat_interleave(self.group, dim=1)\n\n        if past_kv is not None:\n            y = F.scaled_dot_product_attention(\n                q, ka, va,\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n        else:\n            p = int(prefix_len)\n            q_pref, q_suf = q[:, :, :p], q[:, :, p:]\n            k_all, v_all = ka, va\n\n            y_pref = F.scaled_dot_product_attention(\n                q_pref, k_all[:, :, :p], v_all[:, :, :p],\n                attn_mask=None,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            mask = self.suffix_mask[:, :, :q_suf.size(2), :k_all.size(2)]\n            y_suf = F.scaled_dot_product_attention(\n                q_suf, k_all, v_all,\n                attn_mask=mask,\n                dropout_p=0.0,\n                is_causal=False,\n            )\n\n            y = torch.cat([y_pref, y_suf], dim=2)\n\n        y = y.transpose(1, 2).contiguous().view(B, T, D)\n        out = self.o_proj(y)\n\n        return (out, (k, v)) if use_cache else out\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        d, f = cfg["d_model"], cfg["ffn_dim"]\n        self.gate = nn.Linear(d, f, bias=False)\n        self.up = nn.Linear(d, f, bias=False)\n        self.down = nn.Linear(f, d, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass PrefixBlock(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm1 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.attn = PrefixGQAAttention(cfg)\n        self.norm2 = RMSNorm(cfg["d_model"], cfg["norm_eps"])\n        self.ffn = SwiGLU(cfg)\n\n    def forward(self, x, prefix_len, past_kv=None, use_cache=False):\n        if use_cache:\n            a, present = self.attn(\n                self.norm1(x),\n                prefix_len,\n                past_kv=past_kv,\n                use_cache=True,\n            )\n            x = x + a\n            x = x + self.ffn(self.norm2(x))\n            return x, present\n\n        x = x + self.attn(self.norm1(x), prefix_len)\n        x = x + self.ffn(self.norm2(x))\n        return x\n\nclass LegalPrefixFormer145(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = dict(cfg)\n        d = cfg["d_model"]\n\n        self.token_embedding = nn.Embedding(cfg["vocab_size"], d)\n        self.blocks = nn.ModuleList([\n            PrefixBlock(cfg) for _ in range(cfg["n_layers"])\n        ])\n        self.final_norm = RMSNorm(d, cfg["norm_eps"])\n        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)\n\n        if cfg["tie_embeddings"]:\n            self.lm_head.weight = self.token_embedding.weight\n\n        self.apply(self._init_weights)\n\n        residual_scale = 1.0 / math.sqrt(2.0 * cfg["n_layers"])\n        for block in self.blocks:\n            nn.init.normal_(\n                block.attn.o_proj.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n            nn.init.normal_(\n                block.ffn.down.weight,\n                mean=0.0,\n                std=0.02 * residual_scale,\n            )\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if module.bias is not None:\n                nn.init.zeros_(module.bias)\n        elif isinstance(module, nn.Embedding):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n\n    def _run_block(self, block, x, prefix_len):\n        return block(x, prefix_len)\n\n    def forward(\n        self,\n        input_ids,\n        labels=None,\n        prefix_len=None,\n        past_key_values=None,\n        use_cache=False,\n        return_logits=None,\n    ):\n        if prefix_len is None:\n            prefix_len = self.cfg["prefix_length"]\n\n        x = self.token_embedding(input_ids)\n        presents = []\n\n        for i, block in enumerate(self.blocks):\n            pkv = None if past_key_values is None else past_key_values[i]\n\n            if use_cache:\n                x, present = block(\n                    x,\n                    prefix_len,\n                    past_kv=pkv,\n                    use_cache=True,\n                )\n                presents.append(present)\n            elif self.training and self.cfg.get("activation_checkpointing", False):\n                x = activation_checkpoint(\n                    self._run_block,\n                    block,\n                    x,\n                    prefix_len,\n                    use_reentrant=False,\n                )\n            else:\n                x = block(x, prefix_len)\n\n        x = self.final_norm(x)\n\n        loss = None\n        logits = None\n\n        if labels is not None:\n            # Input layout:\n            # prefix[0:1024], BOS[1024], target-input[1025:1280].\n            # Prediction logits at positions [1024:1280] map to 256 labels.\n            target_hidden = x[:, prefix_len:, :]\n            target_logits = self.lm_head(target_hidden)\n            target_labels = labels[:, prefix_len:]\n            loss = F.cross_entropy(\n                target_logits.reshape(-1, target_logits.size(-1)).float(),\n                target_labels.reshape(-1),\n                ignore_index=-100,\n            )\n\n        if return_logits is None:\n            return_logits = labels is None or use_cache\n\n        if return_logits:\n            logits = self.lm_head(x)\n\n        return {\n            "logits": logits,\n            "loss": loss,\n            "past_key_values": tuple(presents) if use_cache else None,\n        }'
ARCH_NS = {}
for _arch, _source in ARCH_SOURCE.items():
    _ns = {"torch":torch, "nn":nn, "F":F, "math":math, "np":np, "checkpoint":checkpoint, "activation_checkpoint":checkpoint}
    exec(compile(_source, f'<phase1_arch_{_arch}>', 'exec'), _ns, _ns)
    if ARCH_CLASS_NAMES[_arch] not in _ns: raise RuntimeError(f'Missing class for architecture {_arch}')
    ARCH_NS[_arch] = _ns
    print(f'Architecture {_arch}: source classes loaded | {ARCH_CLASS_NAMES[_arch]}')
print('All four existing Phase-1 architecture implementations are ready.')

In [ ]:
# View 0 is the first 512 tokens; view 1 is the last 512 tokens when the document is longer.
# Both map back to one original document-level multi-label target.
SEQ_LEN=512; NUM_VIEWS=2; BATCH_SIZE=2; EVAL_BATCH_SIZE=4; TOKEN_CHUNK=250
TOKEN_CACHE=ROOT/"token_cache_v1"; TOKEN_CACHE.mkdir(parents=True,exist_ok=True)

# Dense multi-label targets; label IDs remain the original integer statute IDs at submission time.
def make_target_matrix(records):
    y=np.zeros((len(records),len(LABEL_IDS)),dtype=np.float32)
    for i,r in enumerate(records):
        for lab in r["labels"]: 
            if lab in LABEL_TO_COL: y[i,LABEL_TO_COL[lab]]=1.0
    return y
Y_TRAIN=make_target_matrix(TRAIN_RECORDS)
Y_DEV=make_target_matrix(DEV_RECORDS)
np.save(TOKEN_CACHE/"Y_train.npy",Y_TRAIN); np.save(TOKEN_CACHE/"Y_dev.npy",Y_DEV)

def _record_sig(records,labeled):
    h=hashlib.sha256()
    h.update((DATA_SIGNATURE+f"|n={len(records)}|seq={SEQ_LEN}|views={NUM_VIEWS}|labeled={int(labeled)}").encode())
    return h.hexdigest()

def _tokenize_text(text):
    toks=TOKENIZER.encode(str(text),add_special_tokens=False).ids
    if not toks: toks=[0]
    chunks=[]
    chunks.append(toks[:SEQ_LEN])
    if len(toks)>SEQ_LEN:
        tail=toks[-SEQ_LEN:]
        if tail!=chunks[0]: chunks.append(tail)
    out=np.zeros((NUM_VIEWS,SEQ_LEN),dtype=np.int32)
    mask=np.zeros((NUM_VIEWS,SEQ_LEN),dtype=np.bool_)
    vmask=np.zeros(NUM_VIEWS,dtype=np.bool_)
    for j,chunk in enumerate(chunks[:NUM_VIEWS]):
        n=min(len(chunk),SEQ_LEN); out[j,:n]=chunk[:n]; mask[j,:n]=True; vmask[j]=True
    return out,mask,vmask

def encode_split_cached(records,name,labeled):
    n=len(records); sig=_record_sig(records,labeled)
    xp=TOKEN_CACHE/f"{name}_input_ids.npy"; mp=TOKEN_CACHE/f"{name}_attention_mask.npy"; vp=TOKEN_CACHE/f"{name}_view_mask.npy"; sp=TOKEN_CACHE/f"{name}_token_progress.json"
    compatible=False; start=0
    if xp.exists() and mp.exists() and vp.exists() and sp.exists():
        try:
            st=json.loads(sp.read_text())
            compatible=st.get("signature")==sig and st.get("n")==n and st.get("seq_len")==SEQ_LEN
            if compatible: start=int(st.get("next_index",0))
        except Exception: compatible=False
    if not compatible:
        for p in (xp,mp,vp,sp):
            try:p.unlink()
            except FileNotFoundError:pass
        x=np.lib.format.open_memmap(xp,mode="w+",dtype=np.int32,shape=(n,NUM_VIEWS,SEQ_LEN))
        m=np.lib.format.open_memmap(mp,mode="w+",dtype=np.bool_,shape=(n,NUM_VIEWS,SEQ_LEN))
        v=np.lib.format.open_memmap(vp,mode="w+",dtype=np.bool_,shape=(n,NUM_VIEWS))
        sp.write_text(json.dumps({"signature":sig,"n":n,"seq_len":SEQ_LEN,"next_index":0}))
        start=0
    else:
        x=np.lib.format.open_memmap(xp,mode="r+",dtype=np.int32,shape=(n,NUM_VIEWS,SEQ_LEN))
        m=np.lib.format.open_memmap(mp,mode="r+",dtype=np.bool_,shape=(n,NUM_VIEWS,SEQ_LEN))
        v=np.lib.format.open_memmap(vp,mode="r+",dtype=np.bool_,shape=(n,NUM_VIEWS))
    print(f"{name}: token cache {'resuming' if start else 'building'} at {start:,}/{n:,}",flush=True)
    for i in range(start,n):
        x[i],m[i],v[i]=_tokenize_text(records[i]["text"])
        if (i+1)%TOKEN_CHUNK==0 or i+1==n:
            x.flush();m.flush();v.flush()
            sp.write_text(json.dumps({"signature":sig,"n":n,"seq_len":SEQ_LEN,"next_index":i+1}))
            print(f"{name}: tokenized {i+1:,}/{n:,}",flush=True)
    X=np.load(xp,mmap_mode="r"); M=np.load(mp,mmap_mode="r"); V=np.load(vp,mmap_mode="r")
    return X,M,V

TRAIN_X,TRAIN_M,TRAIN_V=encode_split_cached(TRAIN_RECORDS,"train",True)
DEV_X,DEV_M,DEV_V=encode_split_cached(DEV_RECORDS,"dev",True)
TEST_X,TEST_M,TEST_V=encode_split_cached(TEST_RECORDS,"test",False)
assert TRAIN_X.shape==(len(TRAIN_RECORDS),NUM_VIEWS,SEQ_LEN)
assert DEV_X.shape==(len(DEV_RECORDS),NUM_VIEWS,SEQ_LEN)
assert TEST_X.shape==(len(TEST_RECORDS),NUM_VIEWS,SEQ_LEN)
print("TOKEN CACHE READY:",TRAIN_X.shape,DEV_X.shape,TEST_X.shape,"| label dim:",len(LABEL_IDS))


In [ ]:
ARCH_CLASS_NAMES={"A":"LegalDecoderA","B":"LegalHybrid150","C":"DMLC150","D":"LegalPrefixFormer145"}

class HiddenCaptureHead(nn.Module):
    def __init__(self): super().__init__(); self.last_hidden=None
    def forward(self,hidden): self.last_hidden=hidden; return hidden[..., :1]*0.0

class BottleneckAdapter(nn.Module):
    def __init__(self,dim,bottleneck=32):
        super().__init__(); self.down=nn.Linear(dim,bottleneck); self.act=nn.GELU(); self.up=nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return self.up(self.act(self.down(x)))

class LoRMALinear(nn.Module):
    def __init__(self,base,rank=4,alpha=8.0):
        super().__init__(); self.base=base; self.rank=int(rank); self.alpha=float(alpha)
        for p in self.base.parameters(): p.requires_grad=False
        outdim=base.out_features
        self.A=nn.Parameter(torch.empty(self.rank,outdim)); self.B=nn.Parameter(torch.zeros(outdim,self.rank))
        nn.init.normal_(self.A,mean=0.0,std=0.01)
    def forward(self,x):
        y=self.base(x); return y+F.linear(F.linear(y,self.A),self.B)*(self.alpha/self.rank)

def set_submodule(root,name,module):
    bits=name.split("."); parent=root
    for b in bits[:-1]: parent=getattr(parent,b)
    setattr(parent,bits[-1],module)

def _install_lorma(backbone,rank=4,alpha=8.0):
    targets=[]
    for name,module in list(backbone.named_modules()):
        if not name or not isinstance(module,nn.Linear): continue
        low=name.lower()
        if any(k in low for k in ("attn.","attention.","ffn.","gate.")) and not low.endswith("lm_head"):
            targets.append((name,module))
    for name,module in targets: set_submodule(backbone,name,LoRMALinear(module,rank,alpha))
    if not targets: raise RuntimeError("LoRMA found no supported attention/FFN projections")
    print("LoRMA wrapped linear projections:",len(targets))

def _layer_groups(backbone,arch):
    groups=[]
    if hasattr(backbone,"blocks") and isinstance(backbone.blocks,nn.ModuleList): groups.append(("blocks",list(backbone.blocks)))
    if arch=="C":
        if hasattr(backbone,"encoder"): groups.append(("encoder",list(backbone.encoder)))
        if hasattr(backbone,"decoder"): groups.append(("decoder",list(backbone.decoder)))
    return groups

def _install_adapters(backbone,arch,bottleneck=32,last_n=4):
    groups=_layer_groups(backbone,arch)
    if not groups: raise RuntimeError(f"No supported layer list for adapters on {arch}")
    group,layers=groups[-1]; idx=list(range(max(0,len(layers)-last_n),len(layers)))
    d=int(backbone.cfg["d_model"])
    backbone.lsi_adapter_bank=nn.ModuleDict({str(i):BottleneckAdapter(d,bottleneck) for i in idx})
    handles=[]
    for i in idx:
        def make_hook(key):
            def hook(module,inputs,output):
                adapter=backbone.lsi_adapter_bank[key]
                if isinstance(output,tuple): return (output[0]+adapter(output[0]),)+output[1:]
                return output+adapter(output)
            return hook
        handles.append(layers[i].register_forward_hook(make_hook(str(i))))
    backbone._lsi_adapter_handles=handles
    print(f"Adapters attached to existing {group} layers:",idx)

def _get_raw_state(payload):
    for k in ("model_state_dict","state_dict","model_weights","weights","model"):
        v=payload.get(k) if isinstance(payload,dict) else None
        if isinstance(v,dict) and v and any(torch.is_tensor(z) for z in v.values()): return v
    if isinstance(payload,dict) and payload and all(torch.is_tensor(v) for v in payload.values()): return payload
    raise RuntimeError("Checkpoint has no model state dictionary")

def _clean_state(sd):
    if sd and all(k.startswith("module.") for k in sd): return {k[7:]:v for k,v in sd.items()}
    return sd

def _remap_c_gate_keys(state,model_state):
    out={}; changed=[]
    for k,v in state.items():
        nk=re.sub(r"(\.gate)\.2(?=\.(?:weight|bias)$)",r"\1.1",k)
        if nk!=k and nk in model_state and tuple(model_state[nk].shape)==tuple(v.shape): changed.append((k,nk))
        else: nk=k
        if nk in out: raise RuntimeError(f"Gate key mapping collision: {nk}")
        out[nk]=v
    if changed: print("Compatible C gate key remaps:",changed[:8])
    return out

def load_backbone(arch,method,checkpoint_path=None):
    paths=[Path(checkpoint_path)] if checkpoint_path else list(CHECKPOINT_CANDIDATES.get(arch,[]))
    if not paths: raise FileNotFoundError(f"No Phase-1 checkpoint candidate for architecture {arch}")
    failures=[]
    for path in paths:
        payload=None; model=None
        try:
            try: payload=torch.load(path,map_location="cpu",weights_only=False,mmap=True)
            except (TypeError,RuntimeError,ValueError): payload=torch.load(path,map_location="cpu",weights_only=False)
            cfg=payload_config(payload) or CHECKPOINT_META.get(arch,{}).get("cfg",{})
            if not cfg: raise RuntimeError("Missing model_config; refusing to guess architecture dimensions")
            state=_clean_state(_get_raw_state(payload))
            if _arch_hint(path)!=arch: raise RuntimeError(f"Path architecture hint {_arch_hint(path)} conflicts with requested {arch}")
            cls=ARCH_NS[arch][ARCH_CLASS_NAMES[arch]]
            model=cls(cfg)
            target=model.state_dict()
            if arch=="C": state=_remap_c_gate_keys(state,target)
            missing,unexpected=model.load_state_dict(state,strict=False)
            allowed_missing=[k for k in missing if k.endswith("lm_head.weight") and cfg.get("tie_embeddings",False)]
            bad_missing=[k for k in missing if k not in allowed_missing]
            if bad_missing or unexpected:
                raise RuntimeError(f"Incompatible weights: missing={bad_missing[:8]} unexpected={unexpected[:8]}")
            if arch in "ABD": model.lm_head=HiddenCaptureHead()
            for p in model.parameters(): p.requires_grad=False
            if method=="full":
                for p in model.parameters(): p.requires_grad=True
            elif method=="lorma": _install_lorma(model,rank=4,alpha=8.0)
            elif method in {"partial","partial_C"}:
                groups=_layer_groups(model,arch)
                if arch=="C":
                    for _,layers in groups:
                        for layer in layers[-4:]:
                            for p in layer.parameters(): p.requires_grad=True
                    for nm in ("encoder_final_norm","decoder_final_norm","case_memory_attention","case_memory_norm","case_memory","decoder_start"):
                        mod=getattr(model,nm,None)
                        if isinstance(mod,nn.Parameter): mod.requires_grad=True
                        elif isinstance(mod,nn.Module):
                            for p in mod.parameters(): p.requires_grad=True
                else:
                    if not groups: raise RuntimeError(f"No supported layer groups for partial tuning of {arch}")
                    for _,layers in groups:
                        for layer in layers[-4:]:
                            for p in layer.parameters(): p.requires_grad=True
                    for nm in ("final_norm","encoder_final_norm","decoder_final_norm"):
                        mod=getattr(model,nm,None)
                        if mod is not None:
                            for p in mod.parameters(): p.requires_grad=True
            elif method=="adapters": _install_adapters(model,arch,last_n=4)
            elif method=="bitfit":
                for name,p in model.named_parameters():
                    if name.endswith("bias"): p.requires_grad=True
            else: raise ValueError("Unknown method: "+method)
            model=model.to(DEVICE)
            trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"Loaded Phase-1 {arch} | {path} | d_model={cfg.get('d_model')} | method={method} | trainable={trainable:,}",flush=True)
            del payload,state,target; gc.collect()
            return model,cfg,str(path)
        except Exception as e:
            failures.append(f"{path}: {type(e).__name__}: {str(e)[:300]}")
            print("Checkpoint rejected; trying next candidate:",failures[-1],flush=True)
            try: del payload,model
            except Exception: pass
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    raise RuntimeError(f"No compatible {arch} checkpoint found. Tried {len(failures)} candidates:\n"+"\n".join(failures))

class MultiLabelHead(nn.Module):
    def __init__(self,dim,nlabels):
        super().__init__()
        self.net=nn.Sequential(nn.LayerNorm(3*dim),nn.Dropout(0.2),nn.Linear(3*dim,dim),nn.GELU(),nn.Dropout(0.2),nn.Linear(dim,nlabels))
    def forward(self,x): return self.net(x)

def backbone_features(backbone,arch,input_ids,attention_mask):
    mask=attention_mask.bool()
    if arch=="C":
        memory=backbone.encode_prompt(src_ids=input_ids,src_attention_mask=mask)
        hidden=backbone.decode_teacher_forced(decoder_input_ids=input_ids,memory=memory,decoder_attention_mask=mask)
    else:
        backbone.lm_head.last_hidden=None
        if arch=="D": _=backbone(input_ids=input_ids,prefix_len=input_ids.shape[1],return_logits=True)
        else: _=backbone(input_ids=input_ids)
        hidden=backbone.lm_head.last_hidden
        if hidden is None: raise RuntimeError(f"Hidden capture failed for architecture {arch}")
    h=hidden.float(); m=mask.unsqueeze(-1).float()
    mean=(h*m).sum(dim=1)/m.sum(dim=1).clamp_min(1.0)
    maxv=h.masked_fill(~mask.unsqueeze(-1),-1.0e4).max(dim=1).values
    lengths=mask.long().sum(dim=1).clamp_min(1)-1
    last=h[torch.arange(h.size(0),device=h.device),lengths]
    return torch.cat([mean,maxv,last],dim=-1)

print("Multi-label model utilities ready. Label dimension:",len(LABEL_IDS))


In [ ]:
import copy
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

EPOCHS_PER_EXPERIMENT=2
SESSION_BUDGET_HOURS=7.0
SAVE_EVERY_STEPS=100
PORTABLE_BUNDLE_EVERY=2000
EVAL_EVERY_STEPS=2000
TRAIN_BATCH=2
EVAL_BATCH=4
RUN_START=time.time()
SESSION_DEADLINE=RUN_START+SESSION_BUDGET_HOURS*3600
RESUME_BUNDLE=ROOT/"LSI_PHASE5_RESUME_BUNDLE.zip"
EXPERIMENTS=[
    {"name":"E1_A_Adapters","arch":"A","method":"adapters"},
    {"name":"E2_B_LoRMA","arch":"B","method":"lorma"},
    {"name":"E3_C_PartialLayerwise","arch":"C","method":"partial"},
    {"name":"E4_D_BitFit","arch":"D","method":"bitfit"},
    {"name":"E5_B_FullFineTune","arch":"B","method":"full"},
    {"name":"E6_KnowledgeDistillation","arch":"KD","method":"kd"},
]

def _model_params_state(module,trainable_only=True):
    out={}
    for name,p in module.named_parameters():
        if (not trainable_only) or p.requires_grad: out[name]=p.detach().to("cpu").contiguous().clone()
    return out

def _load_named_params(module,state,strict=True):
    params=dict(module.named_parameters())
    with torch.no_grad():
        for name,value in state.items():
            if name not in params:
                if strict: raise RuntimeError("Parameter missing on resume: "+name)
                continue
            params[name].copy_(value.to(device=params[name].device,dtype=params[name].dtype))
    if strict:
        missing=[n for n,p in params.items() if p.requires_grad and n not in state]
        if missing: raise RuntimeError("Missing trainable params on restore: "+str(missing[:10]))

def _features_for_documents(backbone,arch,X,M,V,indices,batch_device=DEVICE):
    # X/M shape [N, 2, 512]. Evaluate only present views and aggregate chunk vectors to original document.
    raw_x=np.asarray(X[indices],dtype=np.int64)
    raw_m=np.asarray(M[indices],dtype=np.bool_)
    raw_v=np.asarray(V[indices],dtype=np.bool_)
    b=len(indices); view_index=np.argwhere(raw_v)
    ids=torch.as_tensor(raw_x[raw_v],dtype=torch.long,device=batch_device)
    masks=torch.as_tensor(raw_m[raw_v],dtype=torch.bool,device=batch_device)
    if len(ids)==0: raise RuntimeError("A document batch had no valid text views")
    feats=backbone_features(backbone,arch,ids,masks)
    feat_np=feats.new_zeros((b,NUM_VIEWS,feats.shape[-1]))
    feat_np[torch.as_tensor(view_index[:,0],device=batch_device),torch.as_tensor(view_index[:,1],device=batch_device)]=feats
    vmask=torch.as_tensor(raw_v,device=batch_device).unsqueeze(-1).float()
    return (feat_np*vmask).sum(dim=1)/vmask.sum(dim=1).clamp_min(1.0)

def _threshold_metrics(y_true,probs):
    # Select one global threshold on full dev. Macro-F1 is primary; micro-F1/subset accuracy are secondary.
    best=None
    for t in np.arange(0.05,0.951,0.01):
        pred=(probs>=t).astype(np.uint8)
        mf=float(f1_score(y_true,pred,average="macro",zero_division=0))
        candidate=(mf,-abs(float(t)-0.5),float(t),pred)
        if best is None or candidate[:3]>best[:3]: best=candidate
    mf,_,thr,pred=best
    metrics={"macro_f1":float(mf),"micro_f1":float(f1_score(y_true,pred,average="micro",zero_division=0)),
             "macro_precision":float(precision_score(y_true,pred,average="macro",zero_division=0)),
             "macro_recall":float(recall_score(y_true,pred,average="macro",zero_division=0)),
             "subset_accuracy":float(accuracy_score(y_true,pred)),"threshold":float(thr),
             "mean_predicted_labels":float(pred.sum(axis=1).mean()),"mean_gold_labels":float(y_true.sum(axis=1).mean())}
    return metrics,pred

@torch.no_grad()
def _predict_probs(backbone,head,arch,X,M,V,records,name="predict",batch_size=EVAL_BATCH):
    backbone.eval();head.eval(); chunks=[]
    for start in range(0,len(records),batch_size):
        idx=np.arange(start,min(start+batch_size,len(records)))
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            feat=_features_for_documents(backbone,arch,X,M,V,idx)
            logits=head(feat)
        chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
        if (start+len(idx))%1000 < batch_size or start+len(idx)==len(records):
            print(f"{name}: {start+len(idx):,}/{len(records):,}",flush=True)
    return np.concatenate(chunks,axis=0)

def _atomic_torch_save(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+".tmp")
    torch.save(obj,tmp);os.replace(tmp,path)

def _submit_labels(probs,threshold):
    pred=(probs>=float(threshold))
    for i in range(len(pred)):
        if not pred[i].any(): pred[i,int(np.argmax(probs[i]))]=True
    return [[int(LABEL_IDS[j]) for j in np.flatnonzero(row)] for row in pred]

def write_lsi_submission(records,probs,threshold,name):
    pred_ids=_submit_labels(probs,threshold)
    objects=[{"id":r["id"],"pred":list(dict.fromkeys(ids))} for r,ids in zip(records,pred_ids)]
    valid=set(LABEL_IDS)
    assert len(objects)==len(TEST_RECORDS)
    assert [x["id"] for x in objects]==[r["id"] for r in TEST_RECORDS]
    assert all(isinstance(o["pred"],list) and len(o["pred"])>=1 for o in objects)
    assert all(type(lab) is int and lab in valid for o in objects for lab in o["pred"])
    assert all(len(o["pred"])==len(set(o["pred"])) for o in objects)
    run_dir=SUB_DIR/name;run_dir.mkdir(parents=True,exist_ok=True)
    jpath=run_dir/"prediction.json"; zpath=run_dir/"submission.zip"
    tmpj=jpath.with_suffix(".json.tmp");tmpj.write_text(json.dumps(objects,ensure_ascii=False),encoding="utf-8");os.replace(tmpj,jpath)
    tmpz=zpath.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmpz,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=3) as zf: zf.write(jpath,arcname="prediction.json")
    os.replace(tmpz,zpath)
    with zipfile.ZipFile(zpath,"r") as zf:
        assert zf.namelist()==["prediction.json"] and zf.testzip() is None
        check=json.loads(zf.read("prediction.json"))
    assert check==objects
    digest=hashlib.sha256(zpath.read_bytes()).hexdigest()
    print("SUBMISSION SAVED:",zpath,"| SHA256:",digest,flush=True)
    return str(jpath),str(zpath),digest

def _save_best(run_dir,arch,method,basepath,cfg,backbone,head,metrics,dev_probs,step,epoch):
    best={"experiment":run_dir.name,"arch":arch,"method":method,"objective":"knowledge_distillation" if _ACTIVE_KD_TEACHER else method,"teacher_name":_ACTIVE_KD_TEACHER,"base_checkpoint_path":str(basepath),"model_config":cfg,
          "label_ids":LABEL_IDS,"data_signature":DATA_SIGNATURE,"tokenizer_sha256":TOKENIZER_SHA256,
          "best_metrics":metrics,"best_step":int(step),"completed_epochs":int(epoch),"trainable_state_dict":_model_params_state(backbone,True),
          "head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},"seq_len":SEQ_LEN,"num_views":NUM_VIEWS}
    _atomic_torch_save(best,run_dir/"best_model.pt")
    np.save(run_dir/"dev_probabilities.npy",dev_probs.astype(np.float32))
    (run_dir/"best_metrics.json").write_text(json.dumps({k:v for k,v in best.items() if k not in {"model_config","trainable_state_dict","head_state_dict"}},indent=2),encoding="utf-8")
    print("NEW BEST:",run_dir.name,"| macro-F1",round(metrics["macro_f1"],5),"| thr",metrics["threshold"],"| step",step,flush=True)

ACTIVE_EXPERIMENT_NAME=None
_ACTIVE_KD_TEACHER=None

def _bundle_progress():
    tmp=RESUME_BUNDLE.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_STORED) as zf:
        for p in ROOT.rglob("*"):
            if not p.is_file() or p==RESUME_BUNDLE or p==tmp: continue
            rel=p.relative_to(ROOT).as_posix()
            # Token arrays are large but deterministic; they can be rebuilt. Keep only the active optimizer
            # state in the portable bundle; completed runs retain their best task-specific checkpoint and ZIP.
            if rel.startswith("token_cache_v1/"): continue
            if p.name=="resume_state.pt" and p.parent.name!=ACTIVE_EXPERIMENT_NAME: continue
            if p.suffix.lower() in {".pt",".json",".csv",".npy",".zip"}:
                zf.write(p,arcname=rel)
    os.replace(tmp,RESUME_BUNDLE)
    print("PORTABLE RESUME BUNDLE UPDATED:",RESUME_BUNDLE,"| MB",round(RESUME_BUNDLE.stat().st_size/1024**2,1),flush=True)

def train_experiment(exp,teacher_probs=None,teacher_name=None):
    global ACTIVE_EXPERIMENT_NAME, _ACTIVE_KD_TEACHER
    name,arch,method=exp["name"],exp["arch"],exp["method"]
    ACTIVE_EXPERIMENT_NAME=name
    _ACTIVE_KD_TEACHER=teacher_name if teacher_probs is not None else None
    if arch not in ARCH_AVAILABLE: return {"status":"skipped","reason":f"Missing compatible architecture {arch}","experiment":name}
    run_dir=EXP_DIR/name;run_dir.mkdir(parents=True,exist_ok=True)
    resume_path=run_dir/"resume_state.pt";best_path=run_dir/"best_model.pt";history_path=run_dir/"history.csv"
    backbone,cfg,basepath=load_backbone(arch,method)
    head=MultiLabelHead(int(cfg["d_model"]),len(LABEL_IDS)).to(DEVICE)
    # Fail fast before training hours are spent.
    with torch.no_grad():
        pf=_features_for_documents(backbone,arch,TRAIN_X,TRAIN_M,TRAIN_V,np.arange(min(2,len(TRAIN_RECORDS))))
        po=head(pf)
        assert po.shape==(min(2,len(TRAIN_RECORDS)),len(LABEL_IDS)) and torch.isfinite(po).all()
    print("FORWARD SMOKE TEST PASSED:",name,"features",tuple(pf.shape),"logits",tuple(po.shape),flush=True)
    trainable=[p for p in backbone.parameters() if p.requires_grad]
    if not trainable: raise RuntimeError(f"No trainable parameters for {name}")
    backbone_lr={"full":1.5e-5,"lorma":8e-5,"partial":3e-5,"bitfit":5e-4,"adapters":1e-4}[method]
    head_lr=3e-4
    opt=torch.optim.AdamW([{"params":trainable,"lr":backbone_lr},{"params":list(head.parameters()),"lr":head_lr}],weight_decay=0.01)
    batch_size=1 if method=="full" else TRAIN_BATCH
    n=len(TRAIN_RECORDS);steps_per_epoch=math.ceil(n/batch_size);total_steps=steps_per_epoch*EPOCHS_PER_EXPERIMENT
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,total_steps),eta_min=1e-6)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))
    name_seed=int(hashlib.sha1(name.encode("utf-8")).hexdigest()[:8],16)
    order_rng=np.random.default_rng(SEED+name_seed)
    state={"epoch":0,"offset":0,"order":order_rng.permutation(n).astype(np.int32),"global_step":0,"best_f1":-1.0,"stale_evals":0,"history":[],"rng_state":order_rng.bit_generator.state,"completed":False,"experiment":name,"arch":arch,"method":method,"data_signature":DATA_SIGNATURE,"tokenizer_sha256":TOKENIZER_SHA256,"basepath":str(basepath),"epochs_target":EPOCHS_PER_EXPERIMENT,"batch_size":batch_size}
    if resume_path.exists():
        ck=torch.load(resume_path,map_location="cpu",weights_only=False)
        okay=(ck.get("data_signature")==DATA_SIGNATURE and ck.get("tokenizer_sha256")==TOKENIZER_SHA256 and ck.get("experiment")==name and ck.get("arch")==arch and ck.get("method")==method and ck.get("basepath")==str(basepath) and ck.get("epochs_target")==EPOCHS_PER_EXPERIMENT and ck.get("batch_size")==batch_size)
        if not okay: raise RuntimeError(f"Resume file for {name} does not match current data/checkpoint/config. Preserve it and inspect {resume_path}; refusing to overwrite it.")
        _load_named_params(backbone,ck["trainable_state_dict"],strict=True);head.load_state_dict(ck["head_state_dict"])
        opt.load_state_dict(ck["optimizer_state_dict"]);sched.load_state_dict(ck["scheduler_state_dict"])
        try: scaler.load_state_dict(ck["scaler_state_dict"])
        except Exception: pass
        for k in state:
            if k in ck: state[k]=ck[k]
        try: order_rng.bit_generator.state=state["rng_state"]
        except Exception: pass
        print(f"RESUMED {name} | epoch={state['epoch']} offset={state['offset']} step={state['global_step']} best={state['best_f1']:.5f}",flush=True)
    else: print(f"START {name} | arch={arch} | method={method} | train={n:,} | updates/epoch={steps_per_epoch:,} | target epochs={EPOCHS_PER_EXPERIMENT}",flush=True)
    # Positive-label weighting: square-root inverse prevalence, capped to limit instability for rare statutes.
    pos=Y_TRAIN.sum(axis=0); neg=len(Y_TRAIN)-pos
    posw=np.sqrt((neg+1.0)/(pos+1.0));posw=np.clip(posw,1.0,15.0)
    pos_weight=torch.tensor(posw,dtype=torch.float32,device=DEVICE)
    hard_loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    run_start=time.time();last_bundle_step=int(state["global_step"])
    resume_save_every=2000 if method=="full" else SAVE_EVERY_STEPS
    portable_bundle_every=5000 if method=="full" else PORTABLE_BUNDLE_EVERY
    try:
        while int(state["epoch"])<EPOCHS_PER_EXPERIMENT and time.time()<SESSION_DEADLINE:
            off=int(state["offset"]);ordr=np.asarray(state["order"],dtype=np.int32)
            idx=ordr[off:min(off+batch_size,n)]
            if len(idx)==0:
                state["epoch"]+=1;state["offset"]=0
                if state["epoch"]<EPOCHS_PER_EXPERIMENT: state["order"]=order_rng.permutation(n).astype(np.int32);state["rng_state"]=order_rng.bit_generator.state
                continue
            opt.zero_grad(set_to_none=True)
            batch_y=torch.as_tensor(Y_TRAIN[idx],dtype=torch.float32,device=DEVICE)
            with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
                feats=_features_for_documents(backbone,arch,TRAIN_X,TRAIN_M,TRAIN_V,idx)
                logits=head(feats)
                hard=hard_loss_fn(logits.float(),batch_y)
                if teacher_probs is not None:
                    soft=torch.as_tensor(teacher_probs[idx],dtype=torch.float32,device=DEVICE)
                    distill=F.binary_cross_entropy_with_logits(logits.float(),soft)
                    loss=0.55*hard+0.45*distill
                else: loss=hard
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss in {name} at {state['global_step']+1}")
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable+list(head.parameters()),1.0)
            scaler.step(opt);scaler.update();sched.step()
            state["global_step"]+=1;state["last_loss"]=float(loss.detach().cpu());state["offset"]=off+len(idx)
            boundary=state["offset"]>=n
            if boundary:
                state["epoch"]+=1;state["offset"]=0
                if state["epoch"]<EPOCHS_PER_EXPERIMENT:
                    state["order"]=order_rng.permutation(n).astype(np.int32);state["rng_state"]=order_rng.bit_generator.state
                else: state["order"]=np.arange(n,dtype=np.int32)
            step=int(state["global_step"])
            if step%25==0: print(f"{name} | epoch={state['epoch']+1 if not boundary else state['epoch']}/{EPOCHS_PER_EXPERIMENT} | step={step} | rows/epoch={state['offset']:,}/{n:,} | loss={state['last_loss']:.5f} | lr={opt.param_groups[0]['lr']:.2e}",flush=True)
            do_eval=(step%EVAL_EVERY_STEPS==0) or boundary
            if do_eval:
                dev_probs=_predict_probs(backbone,head,arch,DEV_X,DEV_M,DEV_V,DEV_RECORDS,name+" dev")
                metrics,_=_threshold_metrics(Y_DEV,dev_probs)
                state["history"].append({"step":step,"epoch":int(state["epoch"]),"loss":state["last_loss"],**metrics})
                pd.DataFrame(state["history"]).to_csv(history_path,index=False)
                print(f"DEV | {name} | epoch={state['epoch']} step={step} Macro-F1={metrics['macro_f1']:.5f} Micro-F1={metrics['micro_f1']:.5f} SubsetAcc={metrics['subset_accuracy']:.5f} threshold={metrics['threshold']:.2f} avg_pred_labels={metrics['mean_predicted_labels']:.2f}",flush=True)
                if metrics["macro_f1"]>float(state["best_f1"])+1e-6:
                    state["best_f1"]=float(metrics["macro_f1"]);state["best_metrics"]=metrics;state["stale_evals"]=0
                    _save_best(run_dir,arch,method,basepath,cfg,backbone,head,metrics,dev_probs,step,int(state["epoch"]))
                else: state["stale_evals"]+=1
            if step%resume_save_every==0 or do_eval or boundary or (step and step%portable_bundle_every==0):
                payload={**state,"trainable_state_dict":_model_params_state(backbone,True),"head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},"optimizer_state_dict":opt.state_dict(),"scheduler_state_dict":sched.state_dict(),"scaler_state_dict":scaler.state_dict()}
                _atomic_torch_save(payload,resume_path)
            if step and step%portable_bundle_every==0 and step!=last_bundle_step:
                _bundle_progress();last_bundle_step=step
            if step%resume_save_every==0:
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        # Save a resumable state even when the time budget ends mid-epoch.
        payload={**state,"trainable_state_dict":_model_params_state(backbone,True),"head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},"optimizer_state_dict":opt.state_dict(),"scheduler_state_dict":sched.state_dict(),"scaler_state_dict":scaler.state_dict()}
        _atomic_torch_save(payload,resume_path)
        state["completed"]=int(state["epoch"])>=EPOCHS_PER_EXPERIMENT
        (run_dir/"status.json").write_text(json.dumps({"experiment":name,"arch":arch,"method":method,"objective":"knowledge_distillation" if teacher_probs is not None else method,"teacher_name":teacher_name,"completed":state["completed"],"epoch":state["epoch"],"step":state["global_step"],"best_f1":state.get("best_f1"),"data_signature":DATA_SIGNATURE},indent=2),encoding="utf-8")
    except Exception:
        print("TRAINING ERROR — writing emergency resume state:",name,flush=True);traceback.print_exc()
        try:
            payload={**state,"trainable_state_dict":_model_params_state(backbone,True),"head_state_dict":{k:v.detach().cpu().clone() for k,v in head.state_dict().items()},"optimizer_state_dict":opt.state_dict(),"scheduler_state_dict":sched.state_dict(),"scaler_state_dict":scaler.state_dict()}
            _atomic_torch_save(payload,resume_path)
        except Exception as e: print("Emergency checkpoint failed:",repr(e)[:200],flush=True)
        raise
    finally:
        del opt,sched,scaler,backbone,head;gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    completed=int(state["epoch"])>=EPOCHS_PER_EXPERIMENT
    if best_path.exists():
        _generate_submission_from_best(name)
    if completed:
        ACTIVE_EXPERIMENT_NAME=None
        print("EXPERIMENT COMPLETE:",name,"| best Macro-F1",state["best_f1"],flush=True)
    else: print("EXPERIMENT PAUSED SAFELY:",name,"| epoch",state["epoch"],"| step",state["global_step"],flush=True)
    return {"experiment":name,"arch":arch,"method":method,"completed":completed,"best_f1":float(state.get("best_f1",-1.0)),"best_metrics":state.get("best_metrics",{})}

def _generate_submission_from_best(name):
    run_dir=EXP_DIR/name; bestpath=run_dir/"best_model.pt"
    ck=torch.load(bestpath,map_location="cpu",weights_only=False)
    if ck.get("data_signature")!=DATA_SIGNATURE or ck.get("tokenizer_sha256")!=TOKENIZER_SHA256: raise RuntimeError("Best checkpoint signature mismatch")
    arch=ck["arch"]; method=ck["method"]
    backbone,cfg,base=load_backbone(arch,method,ck["base_checkpoint_path"])
    _load_named_params(backbone,ck["trainable_state_dict"],strict=True)
    head=MultiLabelHead(int(cfg["d_model"]),len(LABEL_IDS)).to(DEVICE);head.load_state_dict(ck["head_state_dict"],strict=True)
    probs=_predict_probs(backbone,head,arch,TEST_X,TEST_M,TEST_V,TEST_RECORDS,name+" test")
    np.save(run_dir/"test_probabilities.npy",probs.astype(np.float32))
    p,z,h=write_lsi_submission(TEST_RECORDS,probs,ck["best_metrics"]["threshold"],name)
    shutil.copy2(z,run_dir/"submission.zip")
    (run_dir/"submission_meta.json").write_text(json.dumps({"experiment":name,"arch":arch,"method":method,"objective":ck.get("objective",method),"teacher_name":ck.get("teacher_name"),"dev_metrics":ck["best_metrics"],"zip":str(z),"sha256":h},indent=2),encoding="utf-8")
    del backbone,head;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def _prepare_kd_teacher_targets():
    # Read small JSON metrics first, then load only the winning teacher checkpoint to avoid holding several 600 MB+ models in RAM.
    results=[]
    for mp in EXP_DIR.glob("*/best_metrics.json"):
        try:
            m=json.loads(mp.read_text(encoding="utf-8")); name=mp.parent.name
            if m.get("data_signature")!=DATA_SIGNATURE or not (mp.parent/"best_model.pt").exists(): continue
            score=float(m.get("best_metrics",{}).get("macro_f1",-1.0))
            results.append((score,name,mp.parent/"best_model.pt"))
        except Exception as e: print("KD teacher candidate skipped:",mp,repr(e)[:150])
    if not results: return None,None,None
    results.sort(key=lambda z:z[0],reverse=True)
    teacher_score,teacher_name,teacher_path=results[0]
    teacher_meta=torch.load(teacher_path,map_location="cpu",weights_only=False)
    teacher_arch=teacher_meta["arch"]
    params={"A":243897344,"B":149902080,"C":149061510,"D":144786688}
    smaller=[a for a in ARCH_AVAILABLE if a!=teacher_arch and params.get(a,10**18)<params.get(teacher_arch,0)]
    student_arch=sorted(smaller,key=lambda a:params.get(a,10**18))[0] if smaller else next((a for a in ARCH_AVAILABLE if a!=teacher_arch),None)
    if student_arch is None: print("KD skipped: need a different available student architecture.");return None,None,None
    softpath=ROOT/f"teacher_soft_targets_{teacher_name}.npy"
    if softpath.exists():
        teacher_probs=np.load(softpath)
        if teacher_probs.shape==(len(TRAIN_RECORDS),len(LABEL_IDS)): return teacher_name,student_arch,teacher_probs
    print("Preparing KD teacher:",teacher_name,"arch",teacher_arch,"score",teacher_score,"-> student",student_arch,flush=True)
    backbone,cfg,base=load_backbone(teacher_arch,teacher_meta["method"],teacher_meta["base_checkpoint_path"])
    _load_named_params(backbone,teacher_meta["trainable_state_dict"],strict=True)
    head=MultiLabelHead(int(cfg["d_model"]),len(LABEL_IDS)).to(DEVICE);head.load_state_dict(teacher_meta["head_state_dict"])
    teacher_probs=_predict_probs(backbone,head,teacher_arch,TRAIN_X,TRAIN_M,TRAIN_V,TRAIN_RECORDS,"KD teacher train")
    np.save(softpath,teacher_probs.astype(np.float16))
    del backbone,head;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return teacher_name,student_arch,teacher_probs

def run_experiment_queue():
    global SESSION_DEADLINE
    SESSION_DEADLINE=time.time()+SESSION_BUDGET_HOURS*3600
    records=[]
    result_path=ROOT/"experiment_results.json"
    if result_path.exists():
        try: old=json.loads(result_path.read_text()); records=old.get("results",[])
        except Exception: records=[]
    for exp in EXPERIMENTS:
        name=exp["name"]
        if time.time()>=SESSION_DEADLINE-60:
            print("Session budget reserve reached; stopping before another experiment.",flush=True);break
        if exp["method"]=="kd":
            # KD is runnable only after at least one teacher checkpoint exists.
            teacher_name,student_arch,soft=_prepare_kd_teacher_targets()
            if soft is None:
                print("KD experiment deferred: no compatible completed teacher/student pair yet.",flush=True)
                records.append({"experiment":name,"status":"deferred","reason":"no completed teacher/student pair"})
                break
            exp={"name":name,"arch":student_arch,"method":"adapters"}
            teacher_name_kd=teacher_name
        else: soft=None;teacher_name_kd=None
        status_path=EXP_DIR/name/"status.json"
        if status_path.exists():
            try:
                st=json.loads(status_path.read_text())
                if st.get("completed") and (SUB_DIR/name/"submission.zip").exists():
                    print("Already completed; skip:",name,flush=True);continue
            except Exception: pass
        try:
            res=train_experiment(exp,teacher_probs=soft,teacher_name=teacher_name_kd)
        except Exception as e:
            res={"experiment":name,"arch":exp["arch"],"method":exp["method"],"status":"failed","error":repr(e)}
            print("EXPERIMENT FAILED; preserving earlier outputs and continuing queue:",name,repr(e)[:800],flush=True)
        res["teacher_name"]=teacher_name_kd
        if teacher_name_kd: res["objective"]="knowledge_distillation"
        records=[x for x in records if x.get("experiment")!=name];records.append(res)
        result_path.write_text(json.dumps({"results":records,"available_architectures":ARCH_AVAILABLE,"missing_architectures":MISSING_ARCHS,"data_signature":DATA_SIGNATURE},indent=2),encoding="utf-8")
        if res.get("status")=="skipped":
            continue
        if res.get("status")=="failed" and not (EXP_DIR/name/"resume_state.pt").exists():
            _bundle_progress();continue
        if not res.get("completed",False):
            _bundle_progress();break
        _bundle_progress()
    print("Experiment queue status:")
    if records: print(pd.DataFrame(records).to_string(index=False))
    return records

print("LSI training utilities ready | available architectures:",ARCH_AVAILABLE,"| plan:",[(e['name'],e['arch'],e['method']) for e in EXPERIMENTS])


In [ ]:
# Safe to rerun after a pause. The first incomplete experiment resumes its exact sampler position and optimizer state.
EXPERIMENT_RESULTS=run_experiment_queue()


In [ ]:
from IPython.display import display,FileLink
candidates=[]
for p in SUB_DIR.glob("*/submission_meta.json"):
    try:
        meta=json.loads(p.read_text(encoding="utf-8")); z=Path(meta["zip"])
        if not z.exists(): z=p.parent/"submission.zip"
        if z.exists(): candidates.append((float(meta["dev_metrics"]["macro_f1"]),p.parent.name,z,meta))
    except Exception as e: print("Invalid candidate ignored:",p,repr(e)[:160])
if not candidates:
    print("No completed model submission exists yet. Training likely paused before its first full-dev evaluation. Resume by rerunning Cell 7.")
else:
    candidates.sort(key=lambda x:x[0],reverse=True)
    score,name,bestzip,meta=candidates[0]
    finalzip=SUB_DIR/"submission.zip";finaljson=SUB_DIR/"prediction.json"
    tmp=finalzip.with_suffix(".zip.tmp");shutil.copy2(bestzip,tmp);os.replace(tmp,finalzip)
    with zipfile.ZipFile(finalzip,"r") as zf:
        assert zf.namelist()==["prediction.json"] and zf.testzip() is None
        preds=json.loads(zf.read("prediction.json").decode("utf-8"))
    assert len(preds)==len(TEST_RECORDS)
    assert [str(x["id"]) for x in preds]==[str(r["id"]) for r in TEST_RECORDS]
    valid=set(LABEL_IDS)
    assert all(isinstance(x.get("pred"),list) and len(x["pred"])>=1 for x in preds)
    assert all(type(lab) is int and lab in valid for x in preds for lab in x["pred"])
    assert all(len(x["pred"])==len(set(x["pred"])) for x in preds)
    finaljson.write_text(json.dumps(preds,ensure_ascii=False),encoding="utf-8")
    report={"selected_experiment":name,"development_macro_f1":score,"metrics":meta["dev_metrics"],"source_zip":str(bestzip),"final_zip":str(finalzip),"num_test":len(preds),"label_count":len(LABEL_IDS),"sha256":hashlib.sha256(finalzip.read_bytes()).hexdigest()}
    (ROOT/"final_submission_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print("SELECTED BEST LSI SUBMISSION:\n",json.dumps(report,indent=2))
    display(FileLink(str(finalzip)));display(FileLink(str(finaljson)))

print("\nAVAILABLE OUTPUTS")
for p in [SUB_DIR/"submission.zip",ROOT/"LSI_PHASE5_RESUME_BUNDLE.zip",ROOT/"experiment_results.json",ROOT/"phase5_manifest.json",ROOT/"label_vocabulary.json",ROOT/"final_submission_report.json"]:
    if p.exists():
        print(p,"|",round(p.stat().st_size/1024**2,2),"MB");display(FileLink(str(p)))


In [ ]:
rows=[]
for p in EXP_DIR.glob("*/best_metrics.json"):
    try:
        d=json.loads(p.read_text(encoding="utf-8"))
        row={"experiment":p.parent.name,"arch":d.get("arch"),"method":d.get("method"),"best_step":d.get("best_step"),"epochs":d.get("completed_epochs")}
        row.update(d.get("best_metrics",{})); rows.append(row)
    except Exception as e: print("Could not read",p,repr(e)[:150])
if rows:
    tab=pd.DataFrame(rows).sort_values("macro_f1",ascending=False)
    display(tab)
    tab.to_csv(ROOT/"experiment_comparison.csv",index=False)
    print("Saved comparison:",ROOT/"experiment_comparison.csv")
else: print("No experiment has completed a development evaluation yet.")
print("Missing architecture checkpoints:",MISSING_ARCHS)
print("Required distinct techniques in this plan: adapters, LoRMA, partial/layer-wise fine-tuning, BitFit, full fine-tuning, knowledge distillation.")
print("A skipped/missing/incompatible experiment must be reported as incomplete, not counted as completed.")
